# Notebook 37 · WeatherNext 2 JPCZ ensemble verification

Compare **all 64 members** with ERA5 for **925-hPa horizontal divergence** in the offshore
JPCZ polygon and coastal wedge, at **1, 5, and 10 days**. Negative values indicate
convergence. The two polygons are evaluated independently, including their overlap.

**Catalog** mode processes all selected events. **Pilot** mode processes one event and
reports six ERA5 ranks. Each event contributes one rank per polygon and lead. Synced Drive
checkpoints allow each completed ERA5 target and forecast lead to be reused after a
runtime disconnect.

Run sections 1–5 in this notebook. Its archive reader and rank calculation are included
below. Each helper cell prints a confirmation when it finishes.

The catalog is the same CSV Notebook 35 reads. Notebook 36 supplies the archive reader and
onset timing rule. You can run this notebook directly using that CSV and your approved
WeatherNext account. The plotting batches in Notebooks 35 and 36 are optional.

**New winter events:** start with [Notebook 38 · update and search the catalog](https://colab.research.google.com/github/angelicasophyaramirez-blip/JPCZcatalogcolab/blob/codex/notebook36-weathernext2/notebooks/38_update_winter_catalog.ipynb).
It extends the established Notebook 04/06 detector using frozen thresholds, then writes
the same Drive CSV used here. Reload the catalog and rerun this notebook to update
the ranks and reliability plots. Counts always come from that file; no 275-event limit.

[WeatherNext 2 data and attribution](https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_2_0_0)
· [Rank histogram method, Gneiting et al. (2007)](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jrssb.pdf)
· [Interpretation and case selection, Hamill (2001)](https://doi.org/10.1175/1520-0493%282001%29129%3C0550%3AIORHFV%3E2.0.CO%3B2)


In [ ]:
# @title 1 · Install dependencies in a fresh CPU runtime
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=1.26,<2.4', 'pandas>=2.2,<3', 'xarray>=2026.4,<2027',
                'zarr>=3.1,<4', 'gcsfs>=2025.3,<2027', 'dask[array]>=2025.1,<2027',
                'matplotlib>=3.9,<3.11', 'xlsxwriter>=3.2,<4'], check=True)


In [ ]:
# @title 2 · Connect Google Drive and authenticate for WeatherNext
from google.colab import auth, drive

drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
credentials, detected_project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/cloud-platform'])
print('Drive mounted and credentials acquired. Use the approved account for WN2.')


## 3 · Choose the cases

The study dates match Notebook 36’s provisional 2023-onward window by default. The
operational model’s training cutoff has not been verified. Pilot event **0** selects the
first event in the study window. You can enter an event number that worked in Notebook 36
instead. The inventory displayed below includes the numbers and timing offsets.

The target is the nearest 00, 06, 12, or 18 UTC on the original onset’s UTC date, with
ties going earlier. ERA5 is evaluated at that exact time. The catalog’s peak D12 is kept
as context. If required by your Cloud account, enter your existing project ID.


In [ ]:
# @title 3a · Settings
from pathlib import Path

RUN_MODE = 'Catalog'  # @param ["Pilot", "Catalog"]
PILOT_EVENT_NUMBER = 0  # @param {type:"integer"}
PROJECT_ID = ''  # @param {type:"string"}
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
CATALOG_PATH = DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv'
NOTEBOOK35_DIR = DRIVE_ROOT / 'imerg_manual_case_review'
OUTPUT_DIR = DRIVE_ROOT / 'weathernext2_ensemble_verification'
STUDY_START_UTC = '2023-01-01'  # @param {type:"date"}
STUDY_END_UTC = ''  # @param {type:"string"} # Optional exclusive UTC date.
ALLOW_ERA5T = True  # Use and label preliminary references; refresh them on every rerun.
RUN_OVERRIDES = {}


In [ ]:
# @title 3b · Load WeatherNext archive helpers (in this notebook)
"""WeatherNext 2 single-member atlas, also embedded in standalone Notebook 36.

No credentials, remote listings, or forecast fields are loaded at import time.
Build the notebook with tools/build_weathernext_notebook.py after editing this file.
"""
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
import hashlib
import html
import json
import re

import numpy as np
import pandas as pd
import xarray as xr


LEAD_HOURS = (24, 120, 240)
DATE_COLUMNS = ("event_start", "event_end", "event_peak")
ARCHIVE_ROOT = "weathernext/weathernext_2_0_0/zarr"
SCHEMA_VERSION = 2
PLOT_STYLE_VERSION = 4
VORTEX_LABEL_SCHEME = "notebook35_vortex_screen_v3"
WN2_PRESSURE_LEVELS = (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000)
EARTH_RADIUS_M = 6371000.0
EARTH_OMEGA_S1 = 7.292115e-5
STANDARD_GRAVITY = 9.80665
KAPPA = 2.0 / 7.0
DERIVED_FIELDS = {"divergence", "ertel_pv"}
# The historical merged stores omit variable-level units. Use the published WN2
# schema only for known variables with ABSENT units; never override declared units.
WN2_SCHEMA_UNITS = {
    "geopotential": "m2 s-2", "u_component_of_wind": "m s-1",
    "v_component_of_wind": "m s-1", "temperature": "K",
    "specific_humidity": "kg kg-1", "vertical_velocity": "Pa s-1",
    "mean_sea_level_pressure": "Pa", "2m_temperature": "K",
    "sea_surface_temperature": "K", "total_precipitation_6hr": "m",
    "10m_u_component_of_wind": "m s-1", "10m_v_component_of_wind": "m s-1",
    "100m_u_component_of_wind": "m s-1", "100m_v_component_of_wind": "m s-1",
}
POLYGONS = {
    "JPCZ polygon": [(129.5, 41.0), (136.0, 37.4), (134.5, 35.8), (128.8, 38.0)],
    "Coastal wedge": [(133.05, 35.55), (136.05, 35.55), (139.55, 39.00), (139.55, 42.55)],
}
DOMAINS = {
    "Circulation (110–160°E, 25–60°N)": (110., 160., 25., 60.),
    "Notebook 35 region (125–145°E, 33–47°N)": (125., 145., 33., 47.),
}


@dataclass(frozen=True)
class FieldSpec:
    key: str
    label: str
    variables: tuple[str, ...]
    units: str
    palette: str
    limits: tuple[float, float]
    level: int | None = None
    conversion: str = "identity"

    @property
    def title(self):
        return f"{self.level}-hPa {self.label}" if self.level is not None else self.label


FIELD_PRESETS = {
    "height": FieldSpec("height", "geopotential height", ("geopotential",), "m", "viridis", (4800, 6000), 500, "height"),
    "winds": FieldSpec("winds", "winds", ("u_component_of_wind", "v_component_of_wind"), "m s⁻¹", "YlGnBu", (0, 30), 925, "wind"),
    "divergence": FieldSpec("divergence", "horizontal divergence (derived)", ("u_component_of_wind", "v_component_of_wind"), "10⁻⁵ s⁻¹", "RdBu_r", (-10, 10), 925, "divergence"),
    "ertel_pv": FieldSpec("ertel_pv", "Ertel PV (derived)", ("temperature", "u_component_of_wind", "v_component_of_wind"), "PVU", "RdBu_r", (-2, 2), 1000, "ertel_pv"),
    "temperature": FieldSpec("temperature", "temperature", ("temperature",), "°C", "coolwarm", (-40, 20), 850, "temperature"),
    "humidity": FieldSpec("humidity", "specific humidity", ("specific_humidity",), "g kg⁻¹", "YlGnBu", (0, 10), 925, "humidity"),
    "vertical_velocity": FieldSpec("vertical_velocity", "vertical velocity", ("vertical_velocity",), "Pa s⁻¹", "RdBu_r", (-1, 1), 700),
    "mslp": FieldSpec("mslp", "mean sea-level pressure", ("mean_sea_level_pressure",), "hPa", "viridis", (980, 1040), None, "pressure"),
    "t2m": FieldSpec("t2m", "2-m temperature", ("2m_temperature",), "°C", "coolwarm", (-20, 20), None, "temperature"),
    "winds10m": FieldSpec("winds10m", "10-m winds", ("10m_u_component_of_wind", "10m_v_component_of_wind"), "m s⁻¹", "YlGnBu", (0, 25), None, "wind"),
    "precipitation": FieldSpec("precipitation", "preceding 6-h precipitation", ("total_precipitation_6hr",), "mm / 6 h", "YlGnBu", (0, 30), None, "precipitation"),
}


def utc_naive(value):
    """Use timezone-free UTC for xarray; never interpret a timestamp as local time."""
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        raise ValueError("Missing UTC timestamp.")
    return stamp.tz_localize("UTC").tz_localize(None) if stamp.tzinfo is None else stamp.tz_convert("UTC").tz_localize(None)


def iso_utc(value):
    return utc_naive(value).isoformat() + "Z"


def vortex_classification(record):
    """Use completed event-screen results; legacy manual case labels are unrelated."""
    status = str(record.get("vortex_scan_status", record.get("status", "not_scanned"))).strip().lower()
    if status == "ambiguous":
        return "Vortex status ambiguous"
    if status in {"unavailable", "failed", "error"}:
        return "Vortex status unavailable"
    if status != "complete":
        return "Vortex not checked"
    value = record.get("vortex_detected")
    if isinstance(value, (bool, np.bool_)):
        detected = bool(value)
    elif isinstance(value, (int, float, np.integer, np.floating)) and pd.notna(value) and value in (0, 1):
        detected = bool(value)
    elif isinstance(value, str) and value.strip().lower() in {"true", "false", "0", "1"}:
        detected = value.strip().lower() in {"true", "1"}
    else:
        return "Vortex status unknown"
    return "Vortex" if detected else "Non-vortex"


def vortex_plot_label(record):
    """Use the positive association flag for the two labels shown on figures."""
    if vortex_classification(record) == "Vortex":
        return "Vortex-associated"
    return "Non-vortex-associated"


def vortex_label_metadata(record):
    label = vortex_classification(record)
    detected = True if label == "Vortex" else False if label == "Non-vortex" else None
    return {"vortex_classification": label, "vortex_plot_label": vortex_plot_label(record),
            "vortex_scan_status": str(record.get("vortex_scan_status", "not_scanned")),
            "vortex_detected": detected, "vortex_label_scheme": VORTEX_LABEL_SCHEME,
            "vortex_label_source": str(record.get("vortex_label_source", "saved Notebook 35 event metadata")),
            "vortex_screen_signature": str(record.get("vortex_screen_signature", ""))}


def refresh_vortex_labels(frame, review_dir):
    """Mirror Notebook 35's per-event JSON cache under its active screen signature.

    The inventory CSV can predate a scan. The JSON event records are what Notebook
    35 uses in its event dropdown, so they take precedence, without rerunning scans.
    """
    frame, root = frame.copy(), Path(review_dir) / "vortex_screen"
    active = root / "active_screen.json"
    folder, issue = None, None
    if active.exists():
        try:
            signature = json.loads(active.read_text())["signature"]
            if not isinstance(signature, str) or not re.fullmatch(r"[0-9a-f]{16}", signature):
                raise ValueError("Invalid active vortex-screen signature.")
            folder = root / signature
        except (ValueError, KeyError, OSError, TypeError):
            issue = "unavailable"
    else:
        folders = [p for p in root.glob("*") if p.is_dir() and re.fullmatch(r"[0-9a-f]{16}", p.name)]
        if len(folders) == 1:
            folder = folders[0]
        elif len(folders) > 1:
            issue = "ambiguous"
    records = []
    for _, row in frame.iterrows():
        record = row.to_dict()
        if issue:
            record.update(vortex_scan_status=issue, vortex_detected=None,
                          vortex_label_source="Refresh catalog in updated Notebook 35 to identify its active screen.")
        elif folder is not None:
            path = folder / ("event_" + hashlib.sha256(str(row.review_key).encode()).hexdigest()[:20] + ".json")
            record.update(vortex_scan_status="not_scanned", vortex_detected=None,
                          vortex_label_source=str(path), vortex_screen_signature=folder.name)
            if path.exists():
                try:
                    saved = json.loads(path.read_text())
                    if (not isinstance(saved, dict) or saved.get("status") != "complete" or saved.get("review_key") != row.review_key
                            or saved.get("signature") != folder.name):
                        raise ValueError("Vortex record identity, signature, or completion status does not match.")
                    record.update(vortex_scan_status="complete", vortex_detected=saved.get("vortex_detected"))
                except (ValueError, OSError):
                    record.update(vortex_scan_status="unavailable", vortex_detected=None)
        else:
            record["vortex_label_source"] = "saved Notebook 35 inventory; no event cache folder found"
        records.append(vortex_label_metadata(record))
    updates = pd.DataFrame(records, index=frame.index)
    for column in updates:
        frame[column] = updates[column]
    return frame


def nearest_start_date_time(start):
    """Nearest 00/06/12/18 on the SAME UTC date; equal distances choose earlier."""
    start = utc_naive(start)
    choices = [start.normalize() + pd.Timedelta(hours=h) for h in (0, 6, 12, 18)]
    return min(choices, key=lambda value: (abs(value - start), value))


def normalise_catalog(frame):
    frame = frame.copy()
    for column in DATE_COLUMNS:
        if column not in frame:
            raise ValueError(f"Catalog is missing {column}; use Notebook 35's current merged catalog.")
        frame[column] = pd.to_datetime(frame[column], utc=True, errors="raise").dt.tz_localize(None)
        if frame[column].isna().any():
            raise ValueError(f"Missing timestamps in {column}.")
    if ((frame.event_end < frame.event_start) | (frame.event_peak < frame.event_start)
            | (frame.event_peak > frame.event_end)).any():
        raise ValueError("Each event must satisfy start ≤ peak ≤ end.")
    frame["review_key"] = frame[list(DATE_COLUMNS)].apply(lambda row: "|".join(t.isoformat() for t in row), axis=1)
    if frame.review_key.duplicated().any():
        raise ValueError("Duplicate event start/end/peak identities in catalog.")
    frame = frame.sort_values("event_peak").reset_index(drop=True)
    numbers = pd.to_numeric(frame.get("event_number", pd.Series(index=frame.index, dtype=float)), errors="coerce")
    valid = numbers.notna().all() and numbers.gt(0).all() and numbers.eq(numbers.round()).all() and numbers.is_unique
    frame["event_number"] = numbers.astype(int) if valid else np.arange(1, len(frame) + 1)
    span = (frame.event_end - frame.event_start) / pd.Timedelta(hours=1) + 1
    duration = pd.to_numeric(frame.get("episode_span_hours", frame.get("duration_hours", span)), errors="coerce")
    frame["event_duration_hours"] = duration.where(duration.gt(0)).fillna(span)
    frame["target_valid_time"] = frame.event_start.map(nearest_start_date_time)
    frame["onset_offset_hours"] = (frame.target_valid_time - frame.event_start) / pd.Timedelta(hours=1)
    if "event_peak_D_1e5_s-1" not in frame:
        frame["event_peak_D_1e5_s-1"] = pd.to_numeric(frame.get("event_peak_D_s-1", np.nan), errors="coerce") * 1e5
    return frame


def read_catalog(catalog_path, review_dir, study_start="2023-01-01", study_end=None):
    """Read original inventory; merge reviews only on exact event windows."""
    catalog_path, review_dir = Path(catalog_path), Path(review_dir)
    if not catalog_path.exists():
        raise FileNotFoundError(f"Current catalog not found: {catalog_path}. Check the Drive path used in Notebook 35.")
    catalog = normalise_catalog(pd.read_csv(catalog_path))
    raw_path = catalog_path.parent / "jpcz_catalog_ndjf.csv"
    if raw_path.exists():
        raw = normalise_catalog(pd.read_csv(raw_path))
        missing = [r.event_start for r in raw.itertuples() if not
                   ((catalog.event_start <= r.event_start) & (catalog.event_end >= r.event_end)).any()]
        if missing:
            raise ValueError(f"Merged catalog omits {len(missing)} saved raw fragments. Refresh it with Notebook 06, as in Notebook 35.")
    for filename in ("current_catalog_review_inventory.csv", "case_review_labels.csv"):
        path = review_dir / filename
        if not path.exists():
            continue
        saved = pd.read_csv(path)
        if all(c in saved for c in DATE_COLUMNS):
            for column in DATE_COLUMNS:
                saved[column] = pd.to_datetime(saved[column], utc=True, errors="raise").dt.tz_localize(None)
            saved["review_key"] = saved[list(DATE_COLUMNS)].apply(lambda row: "|".join(t.isoformat() for t in row), axis=1)
        if "review_key" not in saved:
            continue
        columns = [c for c in saved if c.startswith("historical_") or c in
                   ("case_label", "case_notes", "vortex_detected", "vortex_scan_status")]
        if saved.review_key.duplicated().any():
            raise ValueError(f"Duplicate event identities in {path}; resolve before joining reviews.")
        saved = saved.set_index("review_key")
        for column in columns:
            mapped = catalog.review_key.map(saved[column])
            catalog[column] = mapped.combine_first(catalog[column]) if column in catalog else mapped
    catalog = refresh_vortex_labels(catalog, review_dir)
    start = utc_naive(study_start)
    included = catalog.event_start.ge(start)
    reasons = pd.Series("included", index=catalog.index)
    reasons.loc[~included] = "before configured study start (training cutoff unverified)"
    if study_end:
        after = catalog.event_start.ge(utc_naive(study_end))
        included &= ~after
        reasons.loc[after] = "at/after configured study end (exclusive)"
    report = catalog.copy()
    report["selection_status"] = reasons
    report["training_cutoff_status"] = "operational archive training cutoff not verified"
    report["forecast_availability"] = "not checked; use WN2 access check for each event"
    return catalog.loc[included].copy(), report


def forecast_requests(row):
    target = nearest_start_date_time(row["event_start"])
    return pd.DataFrame([{
        "lead_hours": lead, "lead_days": lead // 24,
        "init_time": target - pd.Timedelta(hours=lead), "valid_time": target,
        "event_start": utc_naive(row["event_start"]),
        "onset_offset_hours": (target - utc_naive(row["event_start"])) / pd.Timedelta(hours=1),
    } for lead in LEAD_HOURS])


class GoogleCloudArchive:
    """Metadata-first discovery of WN2 runs, without recursive archive downloads."""
    def __init__(self, credentials, project_id="", root=ARCHIVE_ROOT, overrides=None, *, lazy_indexing=False):
        import gcsfs
        self.root = root.removeprefix("gs://").rstrip("/")
        if self.root != ARCHIVE_ROOT:
            raise ValueError("Use the WN2 ensemble archive root; do not use /success or the mean archive.")
        self.storage_options = {"token": credentials}
        if project_id.strip():
            self.storage_options["project"] = project_id.strip()
        self.fs = gcsfs.GCSFileSystem(**self.storage_options)
        self.lazy_indexing = lazy_indexing
        self.overrides = {iso_utc(k): str(v).removeprefix("gs://").rstrip("/") for k, v in (overrides or {}).items()}
        self.listings = {}
        self.datasets = {}
        self.resolved = {}

    def _list(self, path):
        if path not in self.listings:
            self.listings[path] = self.fs.ls(path, detail=False)
        return self.listings[path]

    def _is_store(self, path):
        return self.fs.exists(path + "/zarr.json") or self.fs.exists(path + "/.zgroup")

    def resolve(self, init_time):
        init_time = utc_naive(init_time)
        identity = iso_utc(init_time)
        if identity in self.resolved:
            return self.resolved[identity]
        if identity in self.overrides:
            candidates = [self.overrides[identity]]
        else:
            partitions = []
            for path in self._list(self.root):
                match = re.fullmatch(r"(\d{4})_to_(\d{4}|present)", path.rstrip("/").rsplit("/", 1)[-1])
                if match and int(match[1]) <= init_time.year <= (9999 if match[2] == "present" else int(match[2])):
                    partitions.append(path.rstrip("/"))
            prefix = init_time.strftime("%Y%m%d_%Hhr")
            candidates = []
            for partition in partitions:
                historical = partition + "/predictions.zarr"
                if self._is_store(historical):
                    # Labels such as 2022_to_2023 use an exclusive upper boundary.
                    # Inspect the actual initialization coordinate, not the label.
                    if contains_initialization(self.open(historical), init_time):
                        candidates.append(historical)
                    continue
                for path in self._list(partition):
                    name = path.rstrip("/").rsplit("/", 1)[-1]
                    if name.startswith(prefix + "_") or name == prefix:
                        candidates.append(path.rstrip("/") + "/predictions.zarr")
        stores = sorted({path for path in candidates if self._is_store(path)})
        if not stores:
            raise FileNotFoundError(f"No WN2 store contains exact initialization {identity}. "
                                    "Checked historical period stores and dated run folders; no other run was substituted.")
        if len(stores) != 1:
            raise ValueError(f"Multiple WN2 stores for {identity}: {stores}. Set RUN_OVERRIDES to an explicit store.")
        if not stores[0].startswith(self.root + "/"):
            raise ValueError("Run overrides must remain within the WN2 ensemble archive.")
        self.resolved[identity] = stores[0]
        return stores[0]

    def open(self, store):
        if store not in self.datasets:
            # xarray/zarr construct an async-capable fsspec store from this URL.
            ds = xr.open_zarr("gs://" + store, storage_options=self.storage_options,
                              chunks=None if self.lazy_indexing else {}, consolidated=None)
            if "sample" not in ds.coords or "sample" not in ds.dims:
                ds.close()
                raise ValueError("Expected the member-resolved WN2 sample coordinate, not ensemble-mean data.")
            # Keep only a few metadata handles; fields remain lazy.
            if len(self.datasets) >= 4:
                self.datasets.pop(next(iter(self.datasets))).close()
            self.datasets[store] = ds
        return self.datasets[store]

    def run(self, init_time):
        store = self.resolve(init_time)
        return store, canonical_run(self.open(store), init_time)

    def clear(self):
        for ds in self.datasets.values():
            ds.close()
        self.datasets.clear()
        self.listings.clear()
        self.resolved.clear()
        self.fs.invalidate_cache()


def contains_initialization(ds, init_time):
    init_time = utc_naive(init_time)
    if "prediction_timedelta" in ds.dims and "time" in ds.coords:
        if not np.issubdtype(ds.time.dtype, np.datetime64):
            raise ValueError("Historical WN2 time must decode to initialization datetimes.")
        return np.datetime64(init_time, "ns") in ds.time.values
    if "init_time" in ds.coords and ds.init_time.size == 1:
        return utc_naive(ds.init_time.values.reshape(-1)[0]) == init_time
    raise ValueError("Unrecognized WN2 initialization-coordinate layout.")


def canonical_run(ds, init_time):
    """Normalize historical and per-run layouts without loading forecast fields."""
    init_time = utc_naive(init_time)
    if not contains_initialization(ds, init_time):
        raise ValueError(f"Store does not contain exact initialization {iso_utc(init_time)}.")
    if "prediction_timedelta" in ds.dims:
        run = ds.sel(time=np.datetime64(init_time, "ns"), drop=True)
        if not np.issubdtype(run.prediction_timedelta.dtype, np.timedelta64):
            raise ValueError("Historical prediction_timedelta must decode to timedeltas.")
        run = run.rename({"prediction_timedelta": "time"})
        if "lead_time_secs" in run.coords:
            expected = run.time.values / np.timedelta64(1, "s")
            if not np.array_equal(run.lead_time_secs.values, expected):
                raise ValueError("Historical lead_time_secs disagrees with prediction_timedelta.")
        run = run.assign_coords(init_time=np.datetime64(init_time, "ns"),
                                datetime=("time", np.datetime64(init_time, "ns") + run.time.values))
        layout = "historical_period_store"
    else:
        run, layout = ds, "dated_run_store"
    run = run.copy(deep=False)
    run.attrs = {**run.attrs, "source_layout": layout}
    for name in run.data_vars:
        attrs = dict(run[name].attrs)
        if not attrs.get("units") and name in WN2_SCHEMA_UNITS:
            attrs.update(units=WN2_SCHEMA_UNITS[name],
                         units_source="official WN2 schema; absent from source variable attributes")
        else:
            attrs["units_source"] = "source variable attributes"
        run[name].attrs = attrs
    return run


def select_forecast(ds, init_time, lead_hours, member, variables, level, domain, *, level_stencil=None):
    """Select an exact lead, level, and region, keeping all samples when member is None."""
    required = {"sample", "time", "lat", "lon", "init_time"}
    if not required.issubset(ds.coords):
        raise ValueError(f"WN2 schema missing coordinates: {sorted(required - set(ds.coords))}.")
    init_time = utc_naive(init_time)
    if ds.init_time.size != 1 or utc_naive(ds.init_time.values.reshape(-1)[0]) != init_time:
        raise ValueError("Store init_time does not match the requested initialization.")
    if not np.issubdtype(ds.time.dtype, np.timedelta64):
        raise ValueError("WN2 time must decode to forecast-lead timedeltas; refusing ambiguous time units.")
    if not (0 < lead_hours <= 360 and lead_hours % 6 == 0):
        raise ValueError("Expected a six-hour lead within the 15-day forecast horizon.")
    lead = pd.Timedelta(hours=lead_hours)
    if member is not None and member not in ds.sample.values:
        raise ValueError(f"Member label {member!r} is not available in this run.")
    if np.timedelta64(lead.value, "ns") not in ds.time.values:
        raise ValueError(f"Exact {lead_hours}-hour lead is unavailable; no nearest forecast was substituted.")
    missing = set(variables) - set(ds.data_vars)
    if missing:
        raise ValueError(f"Requested fields unavailable: {sorted(missing)}.")
    snap = ds[list(variables)].sel(time=lead)
    if member is not None:
        snap = snap.sel(sample=member)
    target = init_time + lead
    if "datetime" in snap.coords and (snap.datetime.size != 1 or utc_naive(snap.datetime.values.reshape(-1)[0]) != target):
        raise ValueError("WN2 datetime is inconsistent with initialization + lead.")
    if level is not None:
        if "level" not in snap.dims or level not in snap.level.values:
            raise ValueError(f"Pressure level {level} hPa is unavailable.")
        if level_stencil is not None:
            if len(level_stencil) != 3 or len(set(level_stencil)) != 3 or level not in level_stencil:
                raise ValueError("PV needs three distinct pressure levels including the target.")
            if not set(level_stencil).issubset(set(snap.level.values)):
                raise ValueError("Requested PV pressure stencil is unavailable.")
            snap = snap.sel(level=list(level_stencil)).sortby("level")
        else:
            snap = snap.sel(level=level)
    elif "level" in snap.dims:
        raise ValueError("Surface preset unexpectedly retains a pressure-level dimension.")
    west, east, south, north = domain
    if not (-180 <= west < east <= 360 and -90 <= south < north <= 90):
        raise ValueError("Invalid geographic domain.")
    # The atlas domains are in East Asia, valid in both common longitude conventions.
    for coordinate, lo, hi in (("lat", south, north), ("lon", west, east)):
        values = np.asarray(snap[coordinate].values)
        if values.ndim != 1 or len(values) < 2:
            raise ValueError(f"Expected a one-dimensional {coordinate} grid.")
        snap = snap.sel({coordinate: slice(lo, hi) if values[0] < values[-1] else slice(hi, lo)})
        if snap.sizes[coordinate] < 2:
            raise ValueError(f"Requested region has too few {coordinate} cells.")
        actual = snap[coordinate].values
        if float(np.min(actual)) > lo + .26 or float(np.max(actual)) < hi - .26:
            raise ValueError(f"Forecast grid does not cover requested {coordinate} bounds.")
    sample_dims = ("sample",) if member is None else ()
    pressure_dims = ("level",) if level_stencil is not None else ()
    order = sample_dims + pressure_dims + ("lat", "lon")
    for name in variables:
        if set(snap[name].dims) != set(order):
            raise ValueError(f"Unexpected remaining dimensions for {name}: {snap[name].dims}.")
    return snap.transpose(*order).sortby("lat").sortby("lon")


def pv_pressure_stencil(available_levels, target):
    levels = sorted({int(p) for p in available_levels})
    if target not in levels or len(levels) < 3:
        raise ValueError("Ertel PV requires the target and at least three available pressure levels.")
    index = levels.index(target)
    first = max(0, min(index - 1, len(levels) - 3))
    stencil = levels[first:first + 3]
    method = "centered" if stencil[0] < target < stencil[-1] else "one-sided at pressure-range boundary"
    return stencil, method


def diagnostic_settings(ds, spec, domain):
    if spec.key not in DERIVED_FIELDS:
        return None
    settings = {"algorithm_version": 2, "numerical_precision": "float64", "earth_radius_m": EARTH_RADIUS_M,
                "horizontal_derivative": "second-order differences on spherical latitude/longitude grid",
                "halo_cells": 2, "below_ground_mask": "not applied; source surface pressure/terrain unavailable",
                "temporal_operation": "instantaneous forecast snapshot"}
    if spec.key == "ertel_pv":
        stencil, method = pv_pressure_stencil(ds.level.values, spec.level)
        settings.update(pressure_stencil_hpa=stencil, vertical_derivative=method,
                        pressure_derivative_units="Pa", theta_reference_hpa=1000,
                        theta_kappa=KAPPA, earth_omega_s1=EARTH_OMEGA_S1,
                        gravity_ms2=STANDARD_GRAVITY, pv_definition="dry hydrostatic pressure-coordinate Ertel PV",
                        formula="-g * (du_dp*dtheta_dy - dv_dp*dtheta_dx + (zeta+f)*dtheta_dp)",
                        display_multiplier=1e6)
    else:
        settings.update(formula="[du/dlambda + d(v*cos(phi))/dphi] / [R*cos(phi)]", display_multiplier=1e5)
    return settings


def diagnostic_domain(ds, domain):
    """Include two source cells outside map bounds when available, then crop after differentiation."""
    lat, lon = ds.lat.values, ds.lon.values
    dlat, dlon = abs(float(lat[1] - lat[0])), abs(float(lon[1] - lon[0]))
    west, east, south, north = domain
    return (max(west - 2 * dlon, float(np.min(lon))), min(east + 2 * dlon, float(np.max(lon))),
            max(south - 2 * dlat, float(np.min(lat))), min(north + 2 * dlat, float(np.max(lat))))


def derive_field(snap, spec, domain):
    """Dry hydrostatic PV and signed horizontal divergence with spherical metric terms.

    u/v are eastward/northward winds. Pressure is in Pa; theta references 1000 hPa.
    PV includes both vertical-shear/horizontal-theta terms, not just (zeta+f)*dtheta/dp.
    """
    for name in spec.variables:
        expected = {"kelvin"} if name == "temperature" else {"wind"}
        if unit_kind(snap[name].attrs.get("units", "")) not in expected:
            raise ValueError(f"Unexpected units for derived-field input {name}: {snap[name].attrs.get('units')!r}.")
    # Only regional inputs are retained; original packed source chunks are decoded serially.
    snap = snap.load(scheduler="synchronous")
    phi = np.deg2rad(np.asarray(snap.lat.values, dtype=float))
    lam = np.deg2rad(np.asarray(snap.lon.values, dtype=float))
    if len(phi) < 3 or len(lam) < 3 or np.any(np.abs(np.cos(phi)) < 1e-6):
        raise ValueError("Diagnostics need at least three latitudes/longitudes away from the poles.")
    if np.any(np.diff(phi) <= 0) or np.any(np.diff(lam) <= 0):
        raise ValueError("Diagnostic latitude/longitude must increase; select_forecast normalizes ordering.")
    cosphi = np.cos(phi)[:, None]
    u = np.asarray(snap.u_component_of_wind.values, dtype=float)
    v = np.asarray(snap.v_component_of_wind.values, dtype=float)
    if spec.key == "divergence":
        values = (np.gradient(u, lam, axis=-1, edge_order=2)
                  + np.gradient(v * cosphi, phi, axis=-2, edge_order=2)) / (EARTH_RADIUS_M * cosphi) * 1e5
        dims = snap.u_component_of_wind.dims
    elif spec.key == "ertel_pv":
        if "sample" in snap.dims:
            raise ValueError("Select one member for Ertel PV.")
        pressure = snap.level.values.astype(float) * 100.0
        if len(pressure) != 3 or np.any(np.diff(pressure) <= 0):
            raise ValueError("PV needs three distinct, increasing pressure levels.")
        theta = np.asarray(snap.temperature.values, dtype=float) * (100000.0 / pressure[:, None, None]) ** KAPPA
        theta_p = np.gradient(theta, pressure, axis=0, edge_order=2)
        u_p = np.gradient(u, pressure, axis=0, edge_order=2)
        v_p = np.gradient(v, pressure, axis=0, edge_order=2)
        theta_x = np.gradient(theta, lam, axis=-1, edge_order=2) / (EARTH_RADIUS_M * cosphi)
        theta_y = np.gradient(theta, phi, axis=-2, edge_order=2) / EARTH_RADIUS_M
        zeta = (np.gradient(v, lam, axis=-1, edge_order=2)
                - np.gradient(u * cosphi, phi, axis=-2, edge_order=2)) / (EARTH_RADIUS_M * cosphi)
        coriolis = 2 * EARTH_OMEGA_S1 * np.sin(phi)[:, None]
        pv = -STANDARD_GRAVITY * (u_p * theta_y - v_p * theta_x + (zeta + coriolis) * theta_p)
        index = list(snap.level.values).index(spec.level)
        values = pv[index] * 1e6
        dims = ("lat", "lon")
    else:
        raise ValueError(f"Unknown derived diagnostic {spec.key}.")
    result = xr.Dataset({"value": (dims, values)}, coords={dim: snap[dim] for dim in dims})
    west, east, south, north = domain
    result = result.sel(lat=slice(south, north), lon=slice(west, east)).reset_coords(drop=True)
    result.value.attrs = {"units": spec.units, "long_name": spec.title}
    return result


def unit_kind(units):
    value = str(units).lower().strip().replace(" ", "").replace("**", "^")
    value = value.replace("²", "2").replace("⁻¹", "-1").replace("⁻²", "-2").replace("^", "")
    value = value.replace("(", "").replace(")", "").replace("·", "")
    groups = {
        "geopotential": {"m2/s2", "m2s-2"}, "height": {"m", "gpm", "geopotentialmetres", "geopotentialmeters"},
        "wind": {"m/s", "ms-1"}, "kelvin": {"k", "kelvin"}, "celsius": {"degc", "celsius", "°c"},
        "humidity": {"kg/kg", "kgkg-1", "1"}, "pa": {"pa"}, "hpa": {"hpa", "mbar", "mb"},
        "vertical_velocity": {"pa/s", "pas-1"}, "millimeters": {"mm"},
    }
    return next((kind for kind, choices in groups.items() if value in choices), None)


def convert_field(snap, spec):
    def field(name, accepted):
        result = snap[name]
        kind = unit_kind(result.attrs.get("units", ""))
        if kind not in accepted:
            raise ValueError(f"Unexpected/missing units for {name}: {result.attrs.get('units')!r}. Check schema before converting.")
        return result, kind

    if spec.conversion == "wind":
        u, _ = field(spec.variables[0], {"wind"})
        v, _ = field(spec.variables[1], {"wind"})
        result = xr.Dataset({"value": np.hypot(u, v), "u": u, "v": v})
    else:
        accepted = {"height": {"geopotential", "height"}, "temperature": {"kelvin", "celsius"},
                    "humidity": {"humidity"}, "pressure": {"pa", "hpa"},
                    "precipitation": {"height", "millimeters"}, "identity": {"vertical_velocity"}}
        value, kind = field(spec.variables[0], accepted[spec.conversion])
        if spec.conversion == "height" and kind == "geopotential":
            value = value / 9.80665
        elif spec.conversion == "temperature" and kind == "kelvin":
            value = value - 273.15
        elif spec.conversion == "humidity":
            value = value * 1000
        elif spec.conversion == "pressure" and kind == "pa":
            value = value / 100
        elif spec.conversion == "precipitation" and kind == "height":
            value = value * 1000
        result = xr.Dataset({"value": value})
    # Cache only spatial fields, not scalar time/level coordinates that differ per row.
    result = result.reset_coords(drop=True)
    result["value"].attrs = {"units": spec.units, "long_name": spec.title}
    if "u" in result:
        result.u.attrs = result.v.attrs = {"units": "m s-1"}
    return result


def cache_identity(store, init_time, lead, member, spec, domain, diagnostic=None):
    identity = {"schema": SCHEMA_VERSION, "store": store, "init_time": iso_utc(init_time),
            "lead_hours": int(lead), "member": str(member), "field": spec.key,
            "variables": list(spec.variables), "level_hpa": spec.level,
            "conversion": spec.conversion, "domain": list(domain)}
    if diagnostic is not None:
        identity["diagnostic"] = diagnostic
    return identity


def load_panel(archive, cache_dir, request, member, spec, domain):
    store, ds = archive.run(request["init_time"])
    diagnostic = diagnostic_settings(ds, spec, domain)
    identity = cache_identity(store, request["init_time"], request["lead_hours"], member, spec, domain, diagnostic)
    encoded = json.dumps(identity, sort_keys=True)
    digest = hashlib.sha256(encoded.encode()).hexdigest()[:24]
    path = Path(cache_dir) / f"wn2_{digest}.nc"
    if path.exists():
        with xr.open_dataset(path) as cached:
            if cached.attrs.get("identity") != encoded:
                raise ValueError(f"Cache identity mismatch: {path}.")
            result = cached.load()
        hit = True
    else:
        selection_domain = diagnostic_domain(ds, domain) if diagnostic else domain
        stencil = diagnostic.get("pressure_stencil_hpa") if diagnostic else None
        snap = select_forecast(ds, request["init_time"], int(request["lead_hours"]), member,
                               spec.variables, spec.level, selection_domain, level_stencil=stencil)
        # Historical chunks include 4 members and all 13 levels over the globe.
        # Decode chunks serially to limit RAM; only the selected regional slice is retained.
        result = derive_field(snap, spec, domain) if diagnostic else convert_field(snap, spec).load(scheduler="synchronous")
        if not np.isfinite(result.value.values).any():
            raise ValueError("Selected forecast field has no finite data in this region.")
        result.attrs["identity"] = encoded
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp.nc")
        result.to_netcdf(temporary)
        temporary.replace(path)
        hit = False
    record = {**identity, "event_number": int(request["event_number"]), "review_key": request["review_key"],
              "event_start": iso_utc(request["event_start"]), "event_end": iso_utc(request["event_end"]),
              "event_peak": iso_utc(request["event_peak"]), "valid_time": iso_utc(request["valid_time"]),
              "onset_offset_hours": float(request["onset_offset_hours"]), "units": spec.units,
              "field_title": spec.title, "display_min": spec.limits[0], "display_max": spec.limits[1],
              "palette": spec.palette, "cache_hit": hit, "cache_file": str(path), "status": "complete",
              "source_layout": ds.attrs.get("source_layout", "documented_single_run_schema"),
              "source_variable_units": json.dumps({name: {"units": ds[name].attrs.get("units"),
                  "units_source": ds[name].attrs.get("units_source", "source variable attributes")} for name in spec.variables}, sort_keys=True),
              "plot_style_version": PLOT_STYLE_VERSION,
              "wind_vector_spacing_degrees": 1 if domain[1] - domain[0] <= 25 else 2,
              "wind_quiver_scale": 350, "wind_key_ms": 10,
              "height_contour_interval_m": (60 if spec.level is not None and spec.level <= 500 else 30) if spec.conversion == "height" else None,
              "training_cutoff_status": "operational archive training cutoff not verified"}
    return {"data": result, "metadata": record, "spec": spec, "domain": tuple(domain)}


def event_panels(archive, cache_dir, row, member, specs, domains, progress=print):
    if len(specs) != 2 or len(domains) != 2:
        raise ValueError("Choose exactly two plot rows.")
    requests = forecast_requests(row)
    panels = []
    for row_index, (spec, domain) in enumerate(zip(specs, domains)):
        for request in requests.to_dict("records"):
            request.update(event_number=int(row.event_number), review_key=row.review_key,
                           event_end=row.event_end, event_peak=row.event_peak)
            progress(f"Row {row_index + 1} | {spec.title} | {request['lead_days']}-day lead | member {member}")
            panel = load_panel(archive, cache_dir, request, member, spec, domain)
            panel["metadata"].update(vortex_label_metadata(row))
            panel["metadata"]["row_index"] = row_index
            panels.append(panel)
    verify_six_panels(panels)
    return panels


def verify_six_panels(panels):
    if len(panels) != 6:
        raise ValueError("A complete event needs exactly six panels.")
    for key in ("member", "valid_time", "review_key"):
        if len({p["metadata"][key] for p in panels}) != 1:
            raise ValueError(f"Six-panel group mixes {key} values.")
    for row in (0, 1):
        group = [p for p in panels if p["metadata"]["row_index"] == row]
        if sorted(p["metadata"]["lead_hours"] for p in group) != list(LEAD_HOURS):
            raise ValueError("Each row must contain exactly the 1-, 5-, and 10-day leads.")
        if len({json.dumps([p["spec"].key, p["spec"].level, p["spec"].limits, p["domain"]]) for p in group}) != 1:
            raise ValueError("Plot settings differ within a row.")
    for panel in panels:
        m = panel["metadata"]
        if utc_naive(m["init_time"]) + pd.Timedelta(hours=m["lead_hours"]) != utc_naive(m["valid_time"]):
            raise ValueError("Initialization + forecast lead does not equal valid time.")


def prepare_map_assets():
    """Resolve Natural Earth files once, before drawing, so network errors are clear."""
    import cartopy.io.shapereader as shpreader
    for category, name in (("physical", "coastline"), ("cultural", "admin_0_boundary_lines_land")):
        shpreader.natural_earth(resolution="110m", category=category, name=name)


def draw_panel(ax, panel, features=True):
    from matplotlib.colors import TwoSlopeNorm
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    spec, ds, metadata = panel["spec"], panel["data"], panel["metadata"]
    crs = ccrs.PlateCarree()
    ax.set_extent(panel["domain"], crs=crs)
    low, high = spec.limits
    if spec.key in DERIVED_FIELDS:
        if not low < 0 < high:
            raise ValueError("Divergence/PV color scales must include negative values, zero, and positive values.")
        levels = np.r_[np.linspace(low, 0, 7), np.linspace(0, high, 7)[1:]]
        norm = TwoSlopeNorm(vmin=low, vcenter=0, vmax=high)
    else:
        levels, norm = np.linspace(low, high, 13), None
    shade = ax.contourf(ds.lon.values, ds.lat.values, ds.value.values, levels=levels,
                        cmap=spec.palette, norm=norm, extend="both", transform=crs)
    if spec.conversion == "height":
        interval = 60 if spec.level is not None and spec.level <= 500 else 30
        lo, hi = float(ds.value.min()), float(ds.value.max())
        contours = np.arange(np.floor(lo / interval) * interval, hi + interval, interval)
        if len(contours) > 1 and hi > lo:
            cs = ax.contour(ds.lon.values, ds.lat.values, ds.value.values,
                            levels=contours, colors="0.2", linewidths=.6, transform=crs)
            ax.clabel(cs, fontsize=7, fmt="%.0f", inline=True)
    if "u" in ds:
        # Regional winds get 1° vectors; the larger circulation map gets 2°.
        spacing = metadata["wind_vector_spacing_degrees"]
        lat_step = max(1, int(round(spacing / abs(float(ds.lat[1] - ds.lat[0])))))
        lon_step = max(1, int(round(spacing / abs(float(ds.lon[1] - ds.lon[0])))))
        sub = ds.isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
        arrows = ax.quiver(sub.lon.values, sub.lat.values, sub.u.values, sub.v.values,
                           transform=crs, angles="uv", scale=metadata["wind_quiver_scale"], width=.003)
        ax.quiverkey(arrows, .82, -.10, metadata["wind_key_ms"], "10 m s⁻¹", labelpos="E", coordinates="axes", fontproperties={"size": 8})
    if features:
        ax.coastlines(resolution="110m", linewidth=.7)
        ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=.4, edgecolor=".4")
    for name, vertices in POLYGONS.items():
        xy = np.asarray(vertices + [vertices[0]])
        ax.plot(xy[:, 0], xy[:, 1], color="darkmagenta" if name == "JPCZ polygon" else "darkorange",
                linewidth=1.2, transform=crs, label=name)
    grid = ax.gridlines(draw_labels=True, linewidth=.35, color=".4", alpha=.5)
    grid.top_labels = grid.right_labels = False
    grid.xlabel_style = grid.ylabel_style = {"size": 8}
    title = (f"Observed event: {vortex_plot_label(metadata)}\n"
                 f"{spec.title} | member {metadata['member']} | {metadata['lead_hours'] // 24}-day lead\n"
                 f"Init: {utc_naive(metadata['init_time']):%Y-%m-%d %H:%M} UTC\n"
                 f"Valid: {utc_naive(metadata['valid_time']):%Y-%m-%d %H:%M} UTC")
    if spec.key == "ertel_pv":
        method = metadata["diagnostic"]["vertical_derivative"]
        stencil = "/".join(map(str, metadata["diagnostic"]["pressure_stencil_hpa"]))
        title += f"\nPV stencil {stencil} hPa · {'one-sided' if method.startswith('one-sided') else 'centered'}"
    ax.set_title(title, fontsize=9, pad=9)
    return shade


def colorbar_label(spec):
    return "10⁻⁵ s⁻¹\nblue: convergence; red: divergence" if spec.key == "divergence" else spec.units


def render_event(panels, row, features=True):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    verify_six_panels(panels)
    fig, axes = plt.subplots(2, 3, figsize=(17, 10.8), subplot_kw={"projection": ccrs.PlateCarree()})
    fig.subplots_adjust(left=.045, right=.91, top=.80, bottom=.12, hspace=.52, wspace=.22)
    for index, panel in enumerate(panels):
        artist = draw_panel(axes.flat[index], panel, features)
        if index % 3 == 2:
            position = axes.flat[index].get_position()
            cax = fig.add_axes([.935, position.y0, .012, position.height])
            fig.colorbar(artist, cax=cax, label=colorbar_label(panel["spec"]))
    m = panels[0]["metadata"]
    peak = pd.to_numeric(row.get("event_peak_D_1e5_s-1", np.nan), errors="coerce")
    fig.suptitle(f"WeatherNext 2 | JPCZ event {int(row.event_number)} | Observed event: {vortex_plot_label(row)} | ensemble member {m['member']}\n"
                 f"Catalog start: {utc_naive(row.event_start):%Y-%m-%d %H:%M} UTC  |  "
                 f"end: {utc_naive(row.event_end):%Y-%m-%d %H:%M} UTC\n"
                 f"Target: {utc_naive(m['valid_time']):%Y-%m-%d %H:%M} UTC  "
                 f"(target − onset: {m['onset_offset_hours']:+g} h)  |  "
                 f"Peak: {utc_naive(row.event_peak):%Y-%m-%d %H:%M} UTC; D12 = {peak:.2f} × 10⁻⁵ s⁻¹",
                 fontsize=12, y=.985, linespacing=1.6)
    axes[0, 0].legend(loc="lower left", fontsize=7, framealpha=.85)
    fig.text(.05, .045, "1 / 5 / 10 days = exact forecast leads to the same target valid time. "
             "Magenta: JPCZ polygon; orange: coastal wedge.\n"
             "Study window is provisional: operational WN2 training cutoff unverified. "
             "Pressure-level fields/diagnostics are unmasked below the surface (especially 1000 hPa).\n"
             "Source: Google DeepMind / Google Research WeatherNext 2 experimental historical data (CC BY 4.0).",
             fontsize=8, linespacing=1.7)
    return fig


def export_event(panels, row, output_dir, figure=None, features=True):
    """Six standalone panels plus the overview, metadata, settings, and attribution."""
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    verify_six_panels(panels)
    metadata = [p["metadata"] for p in panels]
    # Include plotting settings and event identity; cache-hit status is irrelevant.
    stable = [{k: v for k, v in m.items() if k not in ("cache_hit", "cache_file")} for m in metadata]
    signature = hashlib.sha256(json.dumps(stable, sort_keys=True).encode()).hexdigest()[:12]
    m = metadata[0]
    safe_member = re.sub(r"[^A-Za-z0-9_.-]", "_", m["member"])
    destination = Path(output_dir) / (f"event_{int(row.event_number):04d}_start{utc_naive(row.event_start):%Y%m%dT%H%M}"
                                    f"_member{safe_member}_{signature}")
    destination.mkdir(parents=True, exist_ok=True)
    required = [destination / f"overview.{ext}" for ext in ("png", "pdf")]
    for panel in panels:
        item = panel["metadata"]
        stem = f"row{item['row_index'] + 1}_{panel['spec'].key}_lead{item['lead_hours']:03d}h"
        required.extend(destination / f"{stem}.{ext}" for ext in ("png", "pdf"))
    required.extend(destination / name for name in ("forecast_metadata.csv", "settings.json", "attribution.txt"))
    marker = destination / "complete.json"
    if marker.exists() and all(path.exists() and path.stat().st_size > 0 for path in required):
        saved = json.loads(marker.read_text())
        if saved.get("signature") == signature:
            return destination
    marker.unlink(missing_ok=True)
    own_figure = figure is None
    figure = render_event(panels, row, features) if own_figure else figure
    for extension in ("png", "pdf"):
        figure.savefig(destination / f"overview.{extension}", dpi=180, bbox_inches="tight")
    if own_figure:
        plt.close(figure)
    for panel in panels:
        item = panel["metadata"]
        fig, ax = plt.subplots(figsize=(8, 6.8), subplot_kw={"projection": ccrs.PlateCarree()})
        fig.subplots_adjust(bottom=.20, top=.80)
        artist = draw_panel(ax, panel, features)
        fig.colorbar(artist, ax=ax, shrink=.8, pad=.08, label=colorbar_label(panel["spec"]))
        fig.suptitle(f"JPCZ event {int(row.event_number)} | WN2 ensemble member {item['member']}\n"
                     f"Onset {utc_naive(row.event_start):%Y-%m-%d %H:%M} UTC; target − onset {item['onset_offset_hours']:+g} h", fontsize=11)
        fig.text(.08, .04, "Google DeepMind / Google Research WN2 historical data, CC BY 4.0.\n"
                 "Magenta: JPCZ polygon; orange: coastal wedge. Pressure-level fields/diagnostics are unmasked below the surface.\n"
                 "Operational model training cutoff unverified; see metadata and attribution.txt.", fontsize=7)
        stem = f"row{item['row_index'] + 1}_{panel['spec'].key}_lead{item['lead_hours']:03d}h"
        for extension in ("png", "pdf"):
            fig.savefig(destination / f"{stem}.{extension}", dpi=180, bbox_inches="tight")
        plt.close(fig)
    pd.DataFrame(metadata).to_csv(destination / "forecast_metadata.csv", index=False)
    (destination / "settings.json").write_text(json.dumps(stable, indent=2, ensure_ascii=False))
    (destination / "attribution.txt").write_text(
        "© 2025 DeepMind Technologies Limited's machine learning models used to create the experimental data made available at "
        "https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_2_0_0 "
        "under CC BY 4.0 licence terms. This data is intended for experimental modelling only and is not intended, validated, "
        "or approved for real world use.\n\nOperational WN2 model training cutoff has not been verified.\n")
    # A complete marker is written only after all seven figure pairs and metadata exist.
    (destination / "complete.json").write_text(json.dumps({"signature": signature, "forecast_panels": 6, "member": m["member"]}))
    return destination


def find_notebook35_references(review_dir, row):
    records = []
    for path in sorted(Path(review_dir).glob("*_metadata.csv")):
        try:
            saved = pd.read_csv(path)
            if not all(c in saved for c in DATE_COLUMNS):
                continue
            match = pd.Series(True, index=saved.index)
            for column in DATE_COLUMNS:
                match &= pd.to_datetime(saved[column], utc=True, errors="coerce").dt.tz_localize(None).eq(utc_naive(row[column]))
            image = path.with_name(path.name.removesuffix("_metadata.csv") + ".png")
            for item in saved.loc[match].to_dict("records"):
                current = vortex_plot_label(row)
                matches_label = item.get("label_scheme") == VORTEX_LABEL_SCHEME and item.get("label") == current
                records.append({**item, "current_vortex_classification": current, "reference_label_current": matches_label,
                                "metadata_file": str(path), "image_file": str(image) if image.exists() else ""})
        except (ValueError, OSError, pd.errors.ParserError):
            continue
    result = pd.DataFrame(records)
    return result.sort_values("reference_label_current", ascending=False, kind="stable") if len(result) else result



print('Archive helpers loaded. Continue to section 3c.')


In [ ]:
# @title 3c · Load rank and Excel helpers (in this notebook)
"""Onset-aligned verification of WN2 against ERA5 for the two JPCZ regions."""

from __future__ import annotations

from functools import lru_cache
import hashlib
import json
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import xarray as xr
import zarr



ERA5_STORE = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
VERIFICATION_DOMAIN = (125., 145., 33., 47.)
ENSEMBLE_SIZE = 64
DIVERGENCE_SPEC = FIELD_PRESETS["divergence"]
DIVERGENCE_COLUMN = "divergence_1e5_s-1"
RANK_GROUPS = ("1–13", "14–26", "27–39", "40–52", "53–65")
CHECKPOINT_VERSION = 1


def polygon_divergence(field):
    """Area-weighted means on the native grid, independently for each polygon."""
    from matplotlib.path import Path as PolygonPath

    lon, lat = np.meshgrid(field.lon.values, field.lat.values)
    points = np.column_stack((lon.ravel(), lat.ravel()))
    means, counts = {}, {}
    for region, vertices in POLYGONS.items():
        mask = PolygonPath(vertices).contains_points(points).reshape(lon.shape)
        weights = np.cos(np.deg2rad(lat[mask]))
        values = field.values[..., mask]
        if not mask.any() or not np.isfinite(values).all():
            raise ValueError(f"{region} lacks a complete finite set of divergence cells.")
        means[region] = np.average(values, weights=weights, axis=-1)
        counts[region] = int(mask.sum())
    return means, counts


@lru_cache(maxsize=1)
def open_verification_era5():
    # Creating Dask graphs for all decades and variables can exceed Colab RAM
    # before any field is read. Keep backend indexing lazy through subsetting.
    return xr.open_zarr(ERA5_STORE, chunks=None, storage_options={"token": "anon"})


def era5_polygon_values(valid_time, era5, *, allow_era5t=False):
    """Derive divergence from the exact ERA5 925-hPa wind snapshot."""
    target = utc_naive(valid_time)
    stable = era5.attrs.get('valid_time_stop')
    preliminary = era5.attrs.get('valid_time_stop_era5t')
    product = 'ERA5'
    if stable and target > utc_naive(stable):
        if not allow_era5t or not preliminary or target > utc_naive(preliminary):
            raise ValueError(f'ERA5 pending for {iso_utc(target)}; final through {stable}; '
                             f'ERA5T through {preliminary}. No nearest time is substituted.')
        product = 'ERA5T'
    winds = era5[list(DIVERGENCE_SPEC.variables)].sel(time=target, level=DIVERGENCE_SPEC.level)
    winds = winds.rename({"latitude": "lat", "longitude": "lon"})
    west, east, south, north = diagnostic_domain(winds, VERIFICATION_DOMAIN)
    latitude = slice(south, north) if winds.lat[0] < winds.lat[-1] else slice(north, south)
    longitude = slice(west, east) if winds.lon[0] < winds.lon[-1] else slice(east, west)
    winds = winds.sel(lat=latitude, lon=longitude)
    if any(abs(actual - expected) > 1e-5 for actual, expected in zip(
            (float(winds.lon.min()), float(winds.lon.max()), float(winds.lat.min()), float(winds.lat.max())),
            (west, east, south, north))):
        raise ValueError("ERA5 grid does not cover the verification domain and its derivative halo.")
    # Decode one wind variable at a time, only after selecting the region.
    with zarr.config.set({"async.concurrency": 1, "threading.max_workers": 2}):
        for name in DIVERGENCE_SPEC.variables:
            winds[name] = winds[name].load(scheduler="synchronous")
    winds = winds.transpose("lat", "lon").sortby("lat").sortby("lon")
    field = derive_field(winds, DIVERGENCE_SPEC, VERIFICATION_DOMAIN).value
    means, counts = polygon_divergence(field)
    return {"valid_time": iso_utc(target), "source_store": ERA5_STORE, "polygon_cells": counts,
            "reference_product": product, "archive_last_updated": str(era5.attrs.get('last_updated', '')),
            "values": {region: float(value) for region, value in means.items()}}


def forecast_polygon_values(archive, request, progress=print):
    """Read groups of native WN2 samples and immediately reduce each field to two means."""
    store, ds = archive.run(request["init_time"])
    labels = [str(value) for value in ds.sample.values.tolist()]
    if len(labels) != ENSEMBLE_SIZE or len(set(labels)) != ENSEMBLE_SIZE:
        raise ValueError(f"Expected {ENSEMBLE_SIZE} distinct WN2 member labels, found {len(set(labels))}.")
    snap = select_forecast(
        ds, request["init_time"], request["lead_hours"], None, DIVERGENCE_SPEC.variables,
        DIVERGENCE_SPEC.level, diagnostic_domain(ds, VERIFICATION_DOMAIN),
    )
    winds = snap[DIVERGENCE_SPEC.variables[0]]
    sizes = winds.chunksizes.get("sample", (4,) * (ENSEMBLE_SIZE // 4))
    values = {region: [] for region in POLYGONS}
    start = 0
    for size in sizes:
        end = start + size
        progress(f"  Reading {request['lead_days']}-day members {start + 1}–{end} of {ENSEMBLE_SIZE}")
        block = snap.isel(sample=slice(start, end))
        with zarr.config.set({"async.concurrency": 1, "threading.max_workers": 2}):
            for name in DIVERGENCE_SPEC.variables:
                block[name] = block[name].load(scheduler="synchronous")
        field = derive_field(block, DIVERGENCE_SPEC, VERIFICATION_DOMAIN).value
        means, counts = polygon_divergence(field)
        for region in POLYGONS:
            values[region].extend(means[region].tolist())
        start = end
    return {"init_time": iso_utc(request["init_time"]), "valid_time": iso_utc(request["valid_time"]),
            "lead_hours": request["lead_hours"], "source_store": store, "members": labels,
            "polygon_cells": counts, "values": values}


def checkpoint_identity(archive):
    """Reuse verified values only with the same fields, polygons and source configuration."""
    return {
        "version": CHECKPOINT_VERSION, "era5_store": ERA5_STORE, "wn2_root": archive.root,
        "wn2_overrides": archive.overrides, "domain": VERIFICATION_DOMAIN, "polygons": POLYGONS,
        "pressure_hpa": DIVERGENCE_SPEC.level, "ensemble_size": ENSEMBLE_SIZE,
        "diagnostic": diagnostic_settings(None, DIVERGENCE_SPEC, VERIFICATION_DOMAIN),
    }


def atomic_json_file(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as temporary:
        temporary_path = Path(temporary.name)
        try:
            json.dump(payload, temporary, allow_nan=False, indent=2)
        except Exception:
            temporary.close()
            temporary_path.unlink(missing_ok=True)
            raise
    temporary_path.replace(path)


def checkpoint(path, identity, calculate):
    if path.is_file():
        try:
            saved = json.loads(path.read_text())
            if saved["request"] == identity:
                return saved["data"], True
        except (KeyError, ValueError, TypeError):
            pass
    data = calculate()
    atomic_json_file(path, {"request": identity, "data": data})
    return data, False


def observation_rank(members, reference, case_key, *, tie_seed=37):
    """Return the ERA5 rank, its tied interval, and its position relative to the ensemble."""
    values = np.asarray(members, dtype=np.float64)
    if values.shape != (ENSEMBLE_SIZE,) or not np.isfinite(values).all() or not np.isfinite(reference):
        raise ValueError("Ranking requires 64 finite member values and one finite ERA5 value.")
    less = int(np.count_nonzero(values < reference))
    equal = int(np.count_nonzero(values == reference))
    lower, upper = less + 1, less + equal + 1
    key = f"{tie_seed}|{case_key}".encode()
    rng = np.random.default_rng(int.from_bytes(hashlib.sha256(key).digest()[:8], "big"))
    rank = lower if lower == upper else int(rng.integers(lower, upper + 1))
    return {
        "era5_rank": rank, "rank_min_if_tied": lower, "rank_max_if_tied": upper,
        "bin_number": (rank - 1) // 13 + 1,
        "rank_group": RANK_GROUPS[(rank - 1) // 13],
        "below_member_min": bool(reference < values.min()),
        "above_member_max": bool(reference > values.max()),
    }


def case_identity(row):
    return {
        "event_number": int(row.event_number), "review_key": row.review_key,
        "event_start_utc": iso_utc(row.event_start), "event_end_utc": iso_utc(row.event_end),
        "event_peak_utc": iso_utc(row.event_peak), "valid_time_utc": iso_utc(row.target_valid_time),
        "target_minus_onset_hours": float(row.onset_offset_hours),
        "catalog_peak_D12_1e5_s-1": row["event_peak_D_1e5_s-1"],
        "vortex_label": row.get("vortex_plot_label", ""),
        "vortex_scan_status": row.get("vortex_scan_status", ""),
        "vortex_classification": row.get('vortex_classification', vortex_classification(row)),
    }


def run_ensemble_verification(catalog, archive, output_dir, *, era5_reader=open_verification_era5,
                              allow_era5t=False, progress=print):
    """Checkpoint one ERA5 target and each WN2 run/lead independently."""
    identity = checkpoint_identity(archive)
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    cache = Path(output_dir) / "checkpoints" / signature
    member_rows, reference_rows, rank_rows, status_rows = [], [], [], []

    for _, row in catalog.iterrows():
        event = case_identity(row)
        target = utc_naive(row.target_valid_time)
        stamp = target.strftime("%Y%m%dT%H%MZ")
        progress(f"Event {event['event_number']} | verification {event['valid_time_utc']}")
        reference = None
        try:
            progress("  ERA5: opening the reference archive and reading one 925-hPa wind snapshot")
            reference_path = cache / f"era5_{stamp}.json"
            # Preliminary data can change. Refresh it on every run, including when it
            # has become final. Forecast checkpoints remain independent and reusable.
            if reference_path.is_file():
                try:
                    saved = json.loads(reference_path.read_text())
                    if saved['data'].get('reference_product') == 'ERA5T':
                        reference_path.unlink()
                except (KeyError, ValueError, TypeError):
                    pass
            reference, cached = checkpoint(
                reference_path, {"valid_time": event["valid_time_utc"]},
                lambda: era5_polygon_values(target, era5_reader(), allow_era5t=allow_era5t),
            )
            reference_rows.extend([
                {**event, "polygon": region, "pressure_hpa": DIVERGENCE_SPEC.level,
                 DIVERGENCE_COLUMN: reference["values"][region],
                 "polygon_cells": reference["polygon_cells"][region], "source_store": reference["source_store"],
                 "reference_product": reference.get('reference_product', 'ERA5')}
                for region in POLYGONS
            ])
            ref_status = "cached" if cached else "computed"
            ref_error = ""
            progress(f"  ERA5 {ref_status}: polygon means saved")
        except Exception as error:
            reference = None
            ref_status = "failed"
            ref_error = f"{type(error).__name__}: {error}"
            progress(f"  ERA5 unavailable: {ref_error}")

        event_ranks = []
        for request in forecast_requests(row).to_dict("records"):
            lead = request["lead_hours"]
            timing = {"lead_hours": lead, "lead_days": request["lead_days"], "init_time_utc": iso_utc(request["init_time"])}
            status = {**event, **timing, "era5_status": ref_status, "era5_error": ref_error,
                      "reference_product": reference.get('reference_product', 'ERA5') if reference else 'pending'}
            try:
                forecast, cached = checkpoint(
                    cache / f"wn2_{stamp}_lead{lead:03d}h.json",
                    {"init_time": timing["init_time_utc"], "lead_hours": lead, "valid_time": event["valid_time_utc"]},
                    lambda: forecast_polygon_values(archive, request, progress),
                )
                members = forecast["members"]
                if len(members) != ENSEMBLE_SIZE or len(set(members)) != ENSEMBLE_SIZE:
                    raise ValueError("Forecast checkpoint does not contain 64 distinct members.")
                case_values, case_ranks = [], []
                for region in POLYGONS:
                    values = np.asarray(forecast["values"][region], dtype=float)
                    if values.shape != (ENSEMBLE_SIZE,) or not np.isfinite(values).all():
                        raise ValueError(f"{region} does not contain 64 finite member values.")
                    info = {**event, **timing, "polygon": region, "pressure_hpa": DIVERGENCE_SPEC.level}
                    case_values.extend({**info, "member": member, DIVERGENCE_COLUMN: value,
                                        "polygon_cells": forecast["polygon_cells"][region],
                                        "source_store": forecast["source_store"]} for member, value in zip(members, values))
                    if reference is not None:
                        observed = reference["values"][region]
                        rank = observation_rank(values, observed, f"{row.review_key}|{lead}|{region}")
                        case_ranks.append({**info, "member_count": len(members),
                                           "reference_product": reference.get('reference_product', 'ERA5'),
                                           "era5_divergence_1e5_s-1": observed,
                                           "member_min_1e5_s-1": values.min(), "member_mean_1e5_s-1": values.mean(),
                                           "member_max_1e5_s-1": values.max(), **rank})
                member_rows.extend(case_values)
                event_ranks.extend(case_ranks)
                status.update(wn2_status="cached" if cached else "computed", wn2_error="")
            except Exception as error:
                message = f"{type(error).__name__}: {error}"
                status.update(wn2_status="failed", wn2_error=message)
                progress(f"  {request['lead_days']}-day forecast unavailable: {message}")
            status_rows.append(status)
        common_sample = len(event_ranks) == len(LEAD_HOURS) * len(POLYGONS)
        for record in event_ranks:
            record["in_common_sample"] = common_sample
        rank_rows.extend(event_ranks)
    return {
        "case_ranks": pd.DataFrame(rank_rows),
        "member_values": pd.DataFrame(member_rows),
        "era5_reference": pd.DataFrame(reference_rows),
        "batch_status": pd.DataFrame(status_rows),
    }, identity


def rank_histogram_counts(ranks, *, grouped=True):
    """Count every rank, including empty bins, using common cases at each lead."""
    if ranks.empty or not ranks.in_common_sample.any():
        raise ValueError("No event has complete ERA5 and all three WN2 leads.")
    complete = ranks.loc[ranks.in_common_sample]
    n_events = complete.review_key.nunique()
    bin_width = 13 if grouped else 1
    records = []
    for region in POLYGONS:
        for lead in LEAD_HOURS:
            matched = complete.loc[(complete.polygon == region) & (complete.lead_hours == lead)]
            counts = np.bincount(matched.era5_rank.to_numpy(dtype=int), minlength=ENSEMBLE_SIZE + 2)[1:]
            counts = counts.reshape(-1, bin_width).sum(axis=1)
            records.extend({
                "polygon": region, "lead_days": lead // 24, "matched_events": n_events,
                "bin_number": index + 1,
                "rank_from": index * bin_width + 1, "rank_through": (index + 1) * bin_width,
                "case_count": int(count), "uniform_expected_count": n_events * bin_width / (ENSEMBLE_SIZE + 1),
            } for index, count in enumerate(counts))
    return pd.DataFrame(records)


def plot_rank_histograms(ranks, *, grouped=True, title_suffix='All events'):
    """Show either all 65 ranks or five groups, with the same cases at each lead."""
    import matplotlib.pyplot as plt

    bins = rank_histogram_counts(ranks, grouped=grouped)
    complete = ranks.loc[ranks.in_common_sample]
    n_events = complete.review_key.nunique()
    largest_bin = bins.case_count.max()
    expectation = bins.uniform_expected_count.iloc[0]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True, layout="constrained")
    for r, region in enumerate(POLYGONS):
        for c, lead in enumerate(LEAD_HOURS):
            ax = axes[r, c]
            matched = complete.loc[(complete.polygon == region) & (complete.lead_hours == lead)]
            panel_bins = bins.loc[(bins.polygon == region) & (bins.lead_days == lead // 24)]
            positions = ([f"Bin {index + 1}\nRanks {label}" for index, label in enumerate(RANK_GROUPS)]
                         if grouped else np.arange(1, ENSEMBLE_SIZE + 2))
            ax.bar(positions, panel_bins.case_count, color="#277486", width=.8)
            ax.axhline(expectation, color=".25", linestyle="--", linewidth=1)
            ax.set_title(f"{region} · {lead // 24}-day lead · N = {len(matched)}")
            if not grouped:
                ax.set_xticks([1, 9, 17, 25, 33, 41, 49, 57, 65])
            ax.tick_params(axis="x", labelsize=8)
            ax.set_xlabel("ERA5 rank position among 64 forecast values")
            ax.text(.03, .96, f"Outside ensemble range: {matched.below_member_min.sum()} below, "
                    f"{matched.above_member_max.sum()} above", transform=ax.transAxes, va="top", fontsize=8)
            ax.set_ylim(0, 1.22 * max(largest_bin, expectation))
            ax.yaxis.get_major_locator().set_params(integer=True)
            if c == 0:
                ax.set_ylabel("Number of events")
    fig.suptitle(f"{title_suffix} · 925-hPa polygon-average divergence at the onset target · {n_events} matched "
                 f"{'event' if n_events == 1 else 'events'}\n"
                 f"{'13 ranks' if grouped else '1 rank'} per bar · dashed line = uniform rank reference\n"
                 "Lower rank = ERA5 more convergent relative to forecasts · Higher rank = ERA5 less convergent")
    return fig


def ordered_verification_values(members, ranks):
    """Pool 64 forecast values and ERA5, preserving the exact saved tie-broken rank."""
    keys = ["review_key", "polygon", "lead_hours"]
    columns = [*keys, "event_number", "valid_time_utc", "init_time_utc", "lead_days", "pressure_hpa",
               "in_common_sample", "position", "bin_number", "rank_range", "source", "member", DIVERGENCE_COLUMN]
    if ranks.empty:
        return pd.DataFrame(columns=columns)
    if ranks.duplicated(keys).any():
        raise ValueError("Duplicate event/polygon/lead ranks cannot be exported.")
    groups = {key: group for key, group in members.groupby(keys, sort=False)}
    records = []
    for rank in ranks.to_dict("records"):
        key = tuple(rank[name] for name in keys)
        group = groups.get(key)
        if group is None or len(group) != ENSEMBLE_SIZE or group.member.astype(str).nunique() != ENSEMBLE_SIZE:
            raise ValueError(f"Ordered values require 64 distinct members for {key}.")
        values = group[DIVERGENCE_COLUMN].to_numpy(dtype=float)
        observed = rank["era5_divergence_1e5_s-1"]
        if not np.isfinite(values).all() or not np.isfinite(observed):
            raise ValueError(f"Ordered values contain missing or nonfinite divergence for {key}.")
        position = int(rank["era5_rank"])
        lower = int((values < observed).sum()) + 1
        upper = lower + int((values == observed).sum())
        if not lower <= position <= upper:
            raise ValueError(f"Saved ERA5 rank disagrees with the member values for {key}.")
        ordered = group.sort_values(DIVERGENCE_COLUMN, kind="stable")
        entries = [
            {"source": "WN2", "member": str(item["member"]), DIVERGENCE_COLUMN: float(item[DIVERGENCE_COLUMN])}
            for item in ordered.to_dict("records")
        ]
        entries.insert(position - 1, {"source": "ERA5", "member": "", DIVERGENCE_COLUMN: float(observed)})
        info = {name: rank[name] for name in columns[:9]}
        for index, entry in enumerate(entries):
            bin_index = index // 13
            records.append({**info, "position": index + 1, "bin_number": bin_index + 1,
                            "rank_range": RANK_GROUPS[bin_index], **entry})
    return pd.DataFrame(records, columns=columns)


def write_ordered_bins(writer, ordered, ranks):
    """Show thirteen entries in each of five bin columns for every verifiable case."""
    sheet = writer.book.add_worksheet("Ordered bins")
    sheet.hide_gridlines(2)
    sheet.set_zoom(85)
    sheet.freeze_panes(4, 0)
    sheet.set_landscape()
    sheet.fit_to_pages(1, 0)
    book = writer.book
    title_format = book.add_format({"bold": True, "font_size": 13, "font_color": "#17324D"})
    note_format = book.add_format({"text_wrap": True, "valign": "top"})
    bin_format = book.add_format({"bold": True, "bg_color": "#DDECF1", "align": "center"})
    header_format = book.add_format({"bold": True, "text_wrap": True, "bottom": 1})
    value_format = book.add_format({"num_format": "0.000000"})
    era5_value_format = book.add_format({"num_format": "0.000000", "bold": True, "bg_color": "#DDF0DE"})
    era5_label_format = book.add_format({"bold": True, "bg_color": "#DDF0DE"})
    sheet.merge_range(0, 0, 0, 14, "Ordered 925-hPa polygon-average divergence", title_format)
    sheet.merge_range(1, 0, 2, 14,
                      "Each block contains 64 WN2 member values and one ERA5 value, sorted from lowest divergence "
                      "(strongest convergence) to highest divergence. ERA5 is labeled and highlighted. "
                      "Values are in 10⁻⁵ s⁻¹. Histogram counts use only ERA5's saved position.", note_format)
    for bin_index in range(5):
        start = bin_index * 3
        sheet.set_column(start, start, 8)
        sheet.set_column(start + 1, start + 1, 20)
        sheet.set_column(start + 2, start + 2, 16, value_format)
    if ordered.empty:
        sheet.write(4, 0, "No event/polygon/lead has both ERA5 and all 64 members.")
        return
    keys = ["review_key", "polygon", "lead_hours"]
    rank_lookup = {tuple(item[name] for name in keys): item for item in ranks.to_dict("records")}
    row = 4
    for key, group in ordered.groupby(keys, sort=False):
        case = rank_lookup[key]
        sheet.merge_range(row, 0, row, 14,
                          f"Event {case['event_number']} · {case['polygon']} · {case['lead_days']}-day lead · 925 hPa",
                          title_format)
        sheet.merge_range(row + 1, 0, row + 1, 14,
                          f"Valid {case['valid_time_utc']} · Initialized {case['init_time_utc']} · "
                          f"ERA5 rank {case['era5_rank']} of 65 · "
                          f"{'Included in all-lead histograms' if case['in_common_sample'] else 'Incomplete all-lead comparison'}")
        for bin_index, label in enumerate(RANK_GROUPS):
            start = bin_index * 3
            sheet.merge_range(row + 2, start, row + 2, start + 2,
                              f"Bin {bin_index + 1} · ranks {label}", bin_format)
            for offset, text in enumerate(("Position", "Source", "D (10⁻⁵ s⁻¹)")):
                sheet.write(row + 3, start + offset, text, header_format)
        for item in group.to_dict("records"):
            cell_row = row + 4 + (item["position"] - 1) % 13
            cell_col = (item["bin_number"] - 1) * 3
            label_format = era5_label_format if item["source"] == "ERA5" else None
            number_format = era5_value_format if item["source"] == "ERA5" else value_format
            sheet.write_number(cell_row, cell_col, item["position"], label_format)
            source = "ERA5" if item["source"] == "ERA5" else f"WN2 sample {item['member']}"
            sheet.write_string(cell_row, cell_col + 1, source, label_format)
            sheet.write_number(cell_row, cell_col + 2, item[DIVERGENCE_COLUMN], number_format)
        row += 19
    sheet.print_area(0, 0, row - 1, 14)


def export_verification(tables, identity, destination):
    """Write the results and a workbook for inspecting the computed ranks in Drive."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    tables = dict(tables)
    ranks = tables["case_ranks"]
    tables["ordered_values"] = ordered_verification_values(tables["member_values"], ranks)
    # A previously complete run must not leave count tables that describe an old sample.
    for name in ('rank_bin_counts', 'rank_group_counts', 'bin_counts'):
        (destination / f'{name}.csv').unlink(missing_ok=True)
    if not ranks.empty and ranks.in_common_sample.any():
        tables["rank_bin_counts"] = rank_histogram_counts(ranks, grouped=False)
        tables["rank_group_counts"] = rank_histogram_counts(ranks, grouped=True)
        tables["bin_counts"] = tables["rank_group_counts"].pivot(
            index=["polygon", "lead_days", "matched_events"], columns="bin_number", values="case_count"
        ).rename(columns={index + 1: f"Bin {index + 1} · ranks {label}"
                          for index, label in enumerate(RANK_GROUPS)}).reset_index()
    atomic_json_file(destination / "calculation_settings.json", identity)
    for name, frame in tables.items():
        frame.to_csv(destination / f"{name}.csv", index=False)
    with NamedTemporaryFile(suffix=".xlsx", dir=destination, delete=False) as temporary:
        temporary_path = Path(temporary.name)
    try:
        with pd.ExcelWriter(temporary_path, engine="xlsxwriter",
                            engine_kwargs={"options": {"strings_to_formulas": False, "strings_to_urls": False}}) as writer:
            era5_format = writer.book.add_format({"bold": True, "bg_color": "#DDF0DE"})
            for name, frame in tables.items():
                sheet = name.replace("_", " ").capitalize()
                frame.to_excel(writer, sheet_name=sheet, index=False)
                worksheet = writer.sheets[sheet]
                worksheet.freeze_panes(1, 0)
                worksheet.hide_gridlines(2)
                worksheet.set_column(0, max(0, len(frame.columns) - 1), 18)
                if len(frame.columns):
                    worksheet.autofilter(0, 0, len(frame), len(frame.columns) - 1)
                    for index, column in enumerate(frame.columns):
                        width = max(len(column) + 2, 14)
                        if not frame.empty:
                            width = max(width, int(frame[column].head(100).astype(str).str.len().max()) + 2)
                        worksheet.set_column(index, index, min(width, 55))
                if name == "ordered_values" and not frame.empty:
                    from xlsxwriter.utility import xl_col_to_name
                    source_col = xl_col_to_name(frame.columns.get_loc("source"))
                    worksheet.conditional_format(1, 0, len(frame), len(frame.columns) - 1,
                        {"type": "formula", "criteria": "=$" + source_col + '2="ERA5"', "format": era5_format})
                if name == "case_ranks":
                    write_ordered_bins(writer, tables["ordered_values"], ranks)
        temporary_path.replace(destination / "ensemble_verification.xlsx")
    finally:
        temporary_path.unlink(missing_ok=True)
    return destination


def verification_subset(tables, group='All'):
    """Use completed vortex screens for strata; retain unchecked cases as Unknown."""
    ranks = tables['case_ranks'].copy()
    if ranks.empty:
        return ranks
    classification = ranks.get('vortex_classification', pd.Series('', index=ranks.index))
    checked = ranks.vortex_scan_status.eq('complete')
    groups = np.select([checked & classification.eq('Vortex'), checked & classification.eq('Non-vortex')],
                       ['Vortex', 'Non-vortex'], default='Unknown')
    ranks['verification_group'] = groups
    if group not in ('All', 'Vortex', 'Non-vortex', 'Unknown'):
        raise ValueError('Choose All, Vortex, Non-vortex, or Unknown.')
    return ranks if group == 'All' else ranks.loc[ranks.verification_group.eq(group)].copy()


def scheduled_verification_catalog(start, end, *, interval_hours=24, months=(11, 12, 1, 2)):
    """Outcome-independent, half-open UTC schedule including crossings and non-crossings."""
    start, end = utc_naive(start), utc_naive(end)
    if start >= end or interval_hours not in (6, 12, 24):
        raise ValueError('Use start < exclusive end and a 6, 12, or 24 hour schedule.')
    if start != start.floor('6h') or end != end.floor('6h'):
        raise ValueError('Schedule endpoints must be exact six-hour UTC times.')
    times = pd.date_range(start, end, freq=f'{interval_hours}h', inclusive='left')
    times = times[times.month.isin(months)]
    if times.empty:
        raise ValueError('The schedule contains no winter verification targets.')
    frame = pd.DataFrame({'event_start': times, 'event_end': times, 'event_peak': times,
                          'event_number': np.arange(1, len(times) + 1), 'target_valid_time': times,
                          'onset_offset_hours': 0., 'event_peak_D_1e5_s-1': np.nan,
                          'vortex_plot_label': '', 'vortex_scan_status': 'not_applicable'})
    frame['review_key'] = [f'scheduled|{iso_utc(t)}' for t in times]
    return frame


def reliability_probabilities(members, ranks, thresholds, *, sample='event_conditioned'):
    """P(D925 snapshot < threshold) from all 64 members and binary ERA5 outcome.

    This is instantaneous polygon convergence, not the hourly trailing-D12 JPCZ detector.
    Only the common all-lead sample enters, exactly as in the rank histograms.
    """
    if sample not in ('event_conditioned', 'scheduled_winter'):
        raise ValueError('Unknown reliability sampling design.')
    if set(thresholds) != set(POLYGONS) or not np.isfinite(list(thresholds.values())).all():
        raise ValueError('Supply one finite snapshot threshold (10^-5 s^-1) per polygon.')
    keys = ['review_key', 'polygon', 'lead_hours']
    columns = [*keys, 'lead_days', 'valid_time_utc', 'event_number', 'vortex_label', 'vortex_scan_status',
               'sample', 'threshold_1e5_s-1', 'member_count', 'forecast_probability', 'observed', 'brier_score']
    if ranks.empty:
        return pd.DataFrame(columns=columns)
    if ranks.duplicated(keys).any():
        raise ValueError('Duplicate verification cases would overweight reliability.')
    groups = {key: frame for key, frame in members.groupby(keys, sort=False)}
    rows = []
    for row in ranks.loc[ranks.in_common_sample].to_dict('records'):
        key = tuple(row[name] for name in keys)
        frame = groups.get(key)
        if frame is None or len(frame) != ENSEMBLE_SIZE or frame.member.astype(str).nunique() != ENSEMBLE_SIZE:
            raise ValueError(f'Reliability requires 64 distinct members for {key}.')
        values = frame[DIVERGENCE_COLUMN].to_numpy(dtype=float)
        observed_value = float(row['era5_divergence_1e5_s-1'])
        if not np.isfinite(values).all() or not np.isfinite(observed_value):
            raise ValueError('Reliability inputs must be finite.')
        threshold = float(thresholds[row['polygon']])
        probability = float(np.mean(values < threshold))
        observed = int(observed_value < threshold)
        rows.append({**{name: row.get(name, '') for name in columns[:8]},
                     'sample': sample, 'threshold_1e5_s-1': threshold, 'member_count': ENSEMBLE_SIZE,
                     'forecast_probability': probability, 'observed': observed,
                     'brier_score': (probability - observed)**2})
    return pd.DataFrame(rows, columns=columns)


def reliability_bin_counts(probabilities, *, n_bins=10):
    """Fixed [left,right) bins, final bin closed at 1, with empty bins retained."""
    if not isinstance(n_bins, int) or n_bins < 2:
        raise ValueError('Use at least two probability bins.')
    rows = []
    for region in POLYGONS:
        for lead in LEAD_HOURS:
            subset = probabilities.loc[probabilities.polygon.eq(region) & probabilities.lead_hours.eq(lead)]
            if subset.empty:
                continue
            p = subset.forecast_probability.to_numpy(dtype=float)
            y = subset.observed.to_numpy(dtype=float)
            if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any() or not np.isin(y, (0, 1)).all():
                raise ValueError('Need probabilities in [0,1] and binary observed outcomes.')
            bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
            for index in range(n_bins):
                mask = bins == index
                n, hits = int(mask.sum()), int(y[mask].sum())
                frequency = hits / n if n else np.nan
                # Pointwise Wilson 95% binomial intervals; serial dependence is not modeled.
                z = 1.959963984540054
                denominator = 1 + z*z/n if n else np.nan
                center = (frequency + z*z/(2*n)) / denominator if n else np.nan
                half = z * np.sqrt(frequency*(1-frequency)/n + z*z/(4*n*n)) / denominator if n else np.nan
                rows.append({'polygon': region, 'lead_days': lead // 24, 'sample': subset['sample'].iloc[0],
                             'threshold_1e5_s-1': subset['threshold_1e5_s-1'].iloc[0],
                             'bin_number': index + 1, 'probability_from': index/n_bins,
                             'probability_through': (index+1)/n_bins, 'case_count': n, 'observed_count': hits,
                             'mean_forecast_probability': p[mask].mean() if n else np.nan,
                             'observed_frequency': frequency, 'wilson_lower': max(0., center-half) if n else np.nan,
                             'wilson_upper': min(1., center+half) if n else np.nan,
                             'panel_cases': len(subset), 'base_rate': float(y.mean()),
                             'brier_score': float(subset.brier_score.mean())})
    return pd.DataFrame(rows, columns=[
        'polygon', 'lead_days', 'sample', 'threshold_1e5_s-1', 'bin_number',
        'probability_from', 'probability_through', 'case_count', 'observed_count',
        'mean_forecast_probability', 'observed_frequency', 'wilson_lower', 'wilson_upper',
        'panel_cases', 'base_rate', 'brier_score'])


def plot_reliability_diagrams(probabilities, *, n_bins=10, title_suffix='All'):
    """Six reliability panels and inset probability-count histograms."""
    import matplotlib.pyplot as plt

    if probabilities.empty:
        raise ValueError('No complete cases for reliability.')
    counts = reliability_bin_counts(probabilities, n_bins=n_bins)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout='constrained')
    for r, region in enumerate(POLYGONS):
        for c, lead in enumerate(LEAD_HOURS):
            ax = axes[r, c]
            bins = counts.loc[counts.polygon.eq(region) & counts.lead_days.eq(lead // 24)]
            if bins.empty:
                ax.set_title(f'{region} · {lead // 24}-day · No complete cases')
                continue
            filled = bins.loc[bins.case_count.gt(0)]
            x, y = filled.mean_forecast_probability, filled.observed_frequency
            ax.plot([0, 1], [0, 1], '--', color='.4', linewidth=1)
            ax.axhline(bins.base_rate.iloc[0], color='.6', linestyle=':', linewidth=1)
            ax.errorbar(x, y, yerr=np.maximum(0, np.array([y-filled.wilson_lower, filled.wilson_upper-y])),
                        marker='o', color='#277486', capsize=3)
            for px, py, n in zip(x, y, filled.case_count):
                ax.annotate(f'n={n}', (px, py), xytext=(4, 5), textcoords='offset points', fontsize=7)
            ax.set(xlim=(-.03, 1.03), ylim=(-.03, 1.03), xlabel='Forecast probability',
                   ylabel='Observed ERA5 frequency',
                   title=f'{region} · {lead // 24}-day · N={int(bins.panel_cases.iloc[0])}\n'
                         f'D < {bins["threshold_1e5_s-1"].iloc[0]:g} × 10⁻⁵ s⁻¹ · BS={bins.brier_score.iloc[0]:.3f}')
            inset = ax.inset_axes([.58, .10, .35, .22])
            inset.bar((bins.probability_from+bins.probability_through)/2, bins.case_count,
                      width=.9/n_bins, color='#92b8c1')
            inset.set(xlim=(0, 1), title='Forecast counts')
            inset.tick_params(labelsize=6)
            inset.title.set_fontsize(7)
    design = probabilities['sample'].iloc[0]
    description = ('Event-conditioned convergence reliability (ERA5-selected events)'
                   if design == 'event_conditioned' else 'Scheduled winter convergence reliability (crossings and non-crossings)')
    fig.suptitle(f'{description} · {title_suffix}\n'
                 'Instantaneous 925-hPa polygon mean; 64 members; common all-lead cases\n'
                 'Dashed: perfect reliability · dotted: sample base rate · bars: pointwise Wilson 95% (dependence ignored)')
    return fig


def export_reliability(tables, destination, thresholds, *, sample='event_conditioned', group='All', n_bins=10):
    """Refresh probability tables and plots from the current sample, never append old aggregates."""
    import matplotlib.pyplot as plt

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    ranks = verification_subset(tables, group)
    probabilities = reliability_probabilities(tables['member_values'], ranks, thresholds, sample=sample)
    bins = reliability_bin_counts(probabilities, n_bins=n_bins)
    probabilities.to_csv(destination / 'reliability_probabilities.csv', index=False)
    bins.to_csv(destination / 'reliability_bins.csv', index=False)
    if not probabilities.empty:
        summary = probabilities.groupby(['polygon', 'lead_days'], as_index=False).agg(
            cases=('observed', 'size'), base_rate=('observed', 'mean'), brier_score=('brier_score', 'mean'))
        summary.to_csv(destination / 'reliability_summary.csv', index=False)
        fig = plot_reliability_diagrams(probabilities, n_bins=n_bins, title_suffix=group)
        for extension in ('png', 'pdf'):
            fig.savefig(destination / f'reliability_diagrams.{extension}', dpi=180)
        plt.close(fig)
    else:
        # Remove plots from a previous successful run if no current cases remain.
        for name in ('reliability_summary.csv', 'reliability_diagrams.png', 'reliability_diagrams.pdf'):
            (destination / name).unlink(missing_ok=True)
    atomic_json_file(destination / 'reliability_settings.json', {
        'sample': sample, 'group': group, 'thresholds_1e5_s-1': thresholds, 'n_bins': n_bins,
        'criterion': 'instantaneous polygon divergence strictly less than threshold',
        'common_sample': 'ERA5 + 64 WN2 members at every lead and both polygons',
        'not_catalog_D12_event_probability': True,
    })
    return probabilities, bins

print('Verification helpers loaded. Continue to section 3d.')


In [ ]:
# @title 3d · Preview the selected catalog
from IPython.display import display

if not callable(globals().get('read_catalog')) or not callable(globals().get('run_ensemble_verification')):
    raise RuntimeError(
        'Run sections 3b and 3c in this Notebook 37 runtime, then rerun section 3d. '
        'Each helper cell prints a confirmation when it finishes.')
catalog, selection_report = read_catalog(
    CATALOG_PATH, NOTEBOOK35_DIR, STUDY_START_UTC, STUDY_END_UTC or None)
if catalog.empty:
    raise ValueError('The current catalog has no events in the configured study window.')
if RUN_MODE not in ('Pilot', 'Catalog'):
    raise ValueError('Choose Pilot or Catalog.')
if RUN_MODE == 'Pilot':
    selected = catalog.head(1) if PILOT_EVENT_NUMBER == 0 else catalog.loc[
        catalog.event_number.eq(PILOT_EVENT_NUMBER)]
    if selected.empty:
        raise ValueError('Pilot event not found in this study window. Pick a listed event number.')
    result_directory = OUTPUT_DIR / f'pilot_event_{int(selected.event_number.iloc[0]):04d}'
else:
    selected = catalog
    result_directory = OUTPUT_DIR / 'catalog'
archive = GoogleCloudArchive(credentials, PROJECT_ID or detected_project or '',
                             overrides=RUN_OVERRIDES, lazy_indexing=True)
print(f'{len(catalog)} events in the study window. {RUN_MODE} will process {len(selected)}.')
if RUN_MODE == 'Pilot':
    print('Pilot contributes ONE ERA5 rank to each histogram. Choose Catalog to aggregate all selected events.')
print('Result folder:', result_directory)
display(catalog[['event_number', 'event_start', 'event_peak', 'target_valid_time',
                 'onset_offset_hours', 'vortex_plot_label']])


## 4 · Run the verification

Both inputs use the same spherical divergence calculation with a two-grid-cell halo,
and the same cosine-latitude weighting and polygon boundaries on their native grids.
The fixed 925-hPa fields include any below-ground pressure-level values supplied by the
datasets. The coastal wedge results should be interpreted with that in mind. The notebook
reads WN2 members in source chunk groups and saves two means per member and lead.
Archive arrays use lazy indexing. The notebook selects the exact time, pressure level,
and region before reading one wind component at a time, with limited read concurrency.

Spatial averages use every finite grid cell inside each polygon with area weights.
Negative divergence represents convergence. Positive and negative values retain their
signs when averaged and ranked, so the resulting value measures the net convergence
or divergence of the whole polygon.

Each ERA5 rank is one plus the number of member values below ERA5, with seeded random
tie-breaking. Rank 1 means ERA5 was at the most convergent end and rank 65 at the least
convergent end. The exact out-of-range flags are recorded separately from the rank.

The batch status records missing or failed inputs. Rerunning retries those inputs and
reuses completed checkpoints. Cases with ERA5 and all three forecast leads contribute
to the six histograms so that the lead comparison uses the same events.

After a runtime restart, run sections 1–3 again before section 4. Wait for the
confirmations from both helper cells, 3b and 3c, and the catalog preview in 3d.
Completed checkpoints are stored in Drive.


In [ ]:
# @title 4 · Read the selected forecasts and calculate ranks
missing_setup = []
if not callable(globals().get('read_catalog')):
    missing_setup.append('3b · Load WeatherNext archive helpers')
if not callable(globals().get('run_ensemble_verification')):
    missing_setup.append('3c · Load rank and Excel helpers')
if any(name not in globals() for name in ('selected', 'archive', 'OUTPUT_DIR')):
    missing_setup.append('3d · Preview the selected catalog')
if missing_setup:
    raise RuntimeError(
        'Notebook 37 setup is incomplete in this runtime. Run these cells in order, '
        'wait for each to finish, then rerun section 4.\n' + '\n'.join(missing_setup)
        + '\nIf the runtime restarted, first rerun sections 1, 2, and 3a. '
        'Completed Drive checkpoints will be reused.')
# Clear stale archive metadata/listings when retrying a newly published date.
open_verification_era5.cache_clear()
archive.clear()
tables, calculation_settings = run_ensemble_verification(
    selected, archive, OUTPUT_DIR, allow_era5t=ALLOW_ERA5T)
display(tables['batch_status'][['event_number', 'lead_days', 'era5_status', 'wn2_status',
                                'era5_error', 'wn2_error']])
ranks = tables['case_ranks']
complete_events = 0 if ranks.empty else ranks.loc[ranks.in_common_sample, 'review_key'].nunique()
print(f'{len(selected)} selected events; {complete_events} complete events contribute to EACH histogram.')
print(f'{len(selected) - complete_events} events lack at least one required input. Details appear in Batch status.')
if not ranks.empty:
    display(ranks[['event_number', 'valid_time_utc', 'polygon', 'lead_days', 'member_count',
                   'era5_divergence_1e5_s-1', 'era5_rank', 'in_common_sample']])


## 5 · Save and inspect results

The Excel workbook opens in Excel or Google Sheets. The same tables are available as
CSVs. The plots save as PNG and PDF. Pilots and the catalog have separate result folders.

**Case ranks** lists ERA5's value, exact rank, bin number, and ensemble minimum, mean,
and maximum for each event, polygon, and lead. **Member values** retains each forecast
value and native member ID. **ERA5 reference** retains the verifying values.
**Ordered values** is a flat table with all 65 entries per verifiable case, sorted from
most negative divergence to most positive divergence. **Ordered bins** presents the same
numbers in five columns of 13 entries, with ERA5 explicitly labeled and highlighted.
**Bin counts** gives each polygon/lead's event count in each numbered bin.
Values are signed divergence in 10⁻⁵ s⁻¹. Exact rankings use full numerical precision.
Ties retain the same reproducibly chosen ERA5 position as the graphs.

Both plot versions are exported. **65 ranks** shows all possible positions of ERA5
within the ensemble. **5 groups** combines 13 consecutive ranks per bar for readability
with a small event sample. Both use the same calculated ranks and the same set of
complete events. The dashed line shows the expected counts for a uniform rank
distribution. The counts in every panel sum to its displayed N.

A single pilot event populates just one bar per panel. Catalog histograms describe
cases selected around ERA5-detected JPCZ events.


In [ ]:
# @title 5 · Save workbook, tables, and six-panel rank histogram
import matplotlib.pyplot as plt
RANK_DISPLAY = '65 ranks'  # @param ["65 ranks", "5 groups"]

tables['event_selection'] = selection_report
calculation_settings = {
    **calculation_settings, 'catalog': str(CATALOG_PATH), 'study_start_utc': STUDY_START_UTC,
    'study_end_utc_exclusive': STUDY_END_UTC or None, 'run_mode': RUN_MODE,
    'selected_events': len(selected), 'lead_hours': list(LEAD_HOURS), 'tie_seed': 37,
    'allow_era5t': ALLOW_ERA5T,
    'training_cutoff_status': 'operational archive training cutoff unverified',
    'wn2_data_reference': (
        'Google LLC WeatherNext 2 historic experimental data (CC BY 4.0). '
        'https://developers.google.com/earth-engine/datasets/catalog/'
        'projects_gcp-public-data-weathernext_assets_weathernext_2_0_0'
    ),
}
export_verification(tables, calculation_settings, result_directory)
print('Saved', result_directory / 'ensemble_verification.xlsx')
print('CSV tables and both rank-histogram figures are saved in the same folder.')
if not ranks.empty and ranks.in_common_sample.any():
    for grouped, filename in ((False, 'rank_histograms_65ranks'), (True, 'rank_histograms')):
        figure = plot_rank_histograms(ranks, grouped=grouped)
        for extension in ('png', 'pdf'):
            figure.savefig(result_directory / f'{filename}.{extension}', dpi=180)
        if grouped == (RANK_DISPLAY == '5 groups'):
            display(figure)
        plt.close(figure)
else:
    for stem in ('rank_histograms_65ranks', 'rank_histograms'):
        for extension in ('png', 'pdf'):
            (result_directory / f'{stem}.{extension}').unlink(missing_ok=True)
    print('No complete event for all three leads. Check Batch status, then retry section 4.')


## 6 · Reliability diagrams and vortex strata

For each polygon and lead, forecast probability is the fraction of the **64 members**
whose instantaneous 925-hPa polygon-mean divergence is **strictly below** the configured
threshold. ERA5 supplies a binary crossing/non-crossing at the same valid time.
The threshold below is an **editable exploratory snapshot criterion**, in 10⁻⁵ s⁻¹.
Choose it before evaluating the forecast sample. It is **not** the catalog's hourly
trailing-D12 threshold, and these probabilities are not probabilities of the complete
JPCZ detector event or vortex occurrence. Testing a six-hourly approximation to D12
remains a separate scientific step.

These first diagrams use the **ERA5-event-conditioned sample**, the same complete
all-lead cases as the rank histograms. They describe convergence calibration among
selected events, not winter-wide occurrence calibration. Section 7 supplies the
independent winter schedule including non-events for broader assessment.

Fixed probability bins include 0 and 1; empty bins remain in the saved tables but
are not plotted as zero occurrence. Each diagram includes sample counts, base rate,
Brier score, and forecast-count inset. Pointwise Wilson intervals are descriptive;
serially related winter times are not independent uncertainty estimates.

Separate All / Vortex / Non-vortex / Unknown outputs use Notebook 35's completed
screen status. Unchecked or failed screens never enter the Non-vortex stratum.
Changed review labels update the strata without redownloading the forecasts.


In [ ]:
# @title 6 · Refresh event-conditioned reliability and group rank plots
SNAPSHOT_THRESHOLD_1E5 = -2.0  # @param {type:"number"} # Exploratory; choose before analysis.
RELIABILITY_THRESHOLDS = {region: SNAPSHOT_THRESHOLD_1E5 for region in POLYGONS}
PROBABILITY_BINS = 10
for group in ('All', 'Vortex', 'Non-vortex', 'Unknown'):
    folder = result_directory / 'event_conditioned' / group.lower().replace('-', '_')
    probabilities, bins = export_reliability(
        tables, folder, RELIABILITY_THRESHOLDS, group=group, n_bins=PROBABILITY_BINS)
    group_ranks = verification_subset(tables, group)
    n = 0 if group_ranks.empty else group_ranks.loc[group_ranks.in_common_sample, 'review_key'].nunique()
    print(f'{group}: {n} complete events; results in {folder}')
    for grouped, stem in ((False, 'rank_histograms_65ranks'), (True, 'rank_histograms')):
        if n:
            rank_histogram_counts(group_ranks, grouped=grouped).to_csv(folder / f'{stem}_counts.csv', index=False)
            figure = plot_rank_histograms(group_ranks, grouped=grouped, title_suffix=group)
            for extension in ('png', 'pdf'):
                figure.savefig(folder / f'{stem}.{extension}', dpi=180)
            plt.close(figure)
        else:
            for extension in ('png', 'pdf', '_counts.csv'):
                stale = folder / (f'{stem}{extension}' if extension.startswith('_') else f'{stem}.{extension}')
                stale.unlink(missing_ok=True)
    if group == 'All' and not probabilities.empty:
        figure = plot_reliability_diagrams(probabilities, n_bins=PROBABILITY_BINS)
        display(figure)
        plt.close(figure)


## 7 · Scheduled winter reliability, including non-events

This optional retrieval uses a regular UTC schedule fixed independently of observed
events. Both threshold crossings and non-crossings are retained; missing reference or
forecast inputs are marked failed/pending rather than coded as zero. Set the **exclusive**
end to extend the winter sample. By default there is one target per day at 00 UTC in
November–February; choose 6 or 12 hours for a denser schedule. Do not select only dates
with catalog events or select dates according to forecast probabilities.

This can require substantial WN2 reads: every target uses all members at three leads.
The preview is free of field downloads; set RUN_SCHEDULED_RELIABILITY to True to retrieve.
It shares compact wind-derived checkpoints with the catalog run, but saves its own
probabilities, counts, settings, and status table. Its sample contains no event-derived
vortex strata. Training overlap and missing-case selection still limit interpretation.

To extend, rerun this cell with the same start and later end. Failed dates retry;
completed WN2 targets/leads reuse their checkpoints. ERA5T targets refresh until final.
A November 19 target requests initializations on November 18, 14, and 9 at the same UTC
hour for the 1-, 5-, and 10-day leads. Exact WN2 availability is checked on each request;
the workflow never substitutes a later run or another model.


In [ ]:
# @title 7 · Preview or run the outcome-independent winter schedule
RUN_SCHEDULED_RELIABILITY = False  # @param {type:"boolean"}
SCHEDULE_START_UTC = '2026-11-01'  # @param {type:"date"}
SCHEDULE_END_UTC = '2027-03-01'  # @param {type:"date"} # Exclusive; future times stay pending.
SCHEDULE_INTERVAL_HOURS = 24  # @param [6, 12, 24] {type:"raw"}
scheduled = scheduled_verification_catalog(
    SCHEDULE_START_UTC, SCHEDULE_END_UTC, interval_hours=SCHEDULE_INTERVAL_HOURS)
print(f'{len(scheduled)} winter targets; {len(scheduled)*len(LEAD_HOURS)} forecast requests.')
display(scheduled[['event_number', 'target_valid_time']])
if RUN_SCHEDULED_RELIABILITY:
    # Future schedule entries remain visible in the preview, but are not downloaded.
    due = scheduled.loc[scheduled.target_valid_time.le(pd.Timestamp.now(tz='UTC').tz_localize(None))]
    if due.empty:
        raise ValueError('No schedule targets are due yet.')
    open_verification_era5.cache_clear()
    archive.clear()
    scheduled_tables, scheduled_settings = run_ensemble_verification(
        due, archive, OUTPUT_DIR, allow_era5t=ALLOW_ERA5T)
    schedule_dir = OUTPUT_DIR / 'scheduled_winter'
    schedule_dir.mkdir(parents=True, exist_ok=True)
    for name, frame in scheduled_tables.items():
        frame.to_csv(schedule_dir / f'{name}.csv', index=False)
    schedule_settings = {**scheduled_settings, 'start': SCHEDULE_START_UTC,
        'end_exclusive': SCHEDULE_END_UTC, 'interval_hours': SCHEDULE_INTERVAL_HOURS,
        'due_targets': len(due), 'future_targets': len(scheduled)-len(due), 'allow_era5t': ALLOW_ERA5T}
    atomic_json_file(schedule_dir / 'schedule_settings.json', schedule_settings)
    scheduled_probabilities, scheduled_bins = export_reliability(
        scheduled_tables, schedule_dir, RELIABILITY_THRESHOLDS,
        sample='scheduled_winter', n_bins=PROBABILITY_BINS)
    display(scheduled_tables['batch_status'])
    verified = scheduled_probabilities.review_key.nunique() if not scheduled_probabilities.empty else 0
    print(f'{verified}/{len(due)} due targets verified at all leads; {len(scheduled)-len(due)} future targets.')
    if not scheduled_probabilities.empty:
        figure = plot_reliability_diagrams(scheduled_probabilities, n_bins=PROBABILITY_BINS)
        display(figure)
        plt.close(figure)
